# Aegis v2: artifact-resistance and sensitivity audit

This run audits the Random Forest baseline. **It is not an unbiased estimate of final
generalization:** the baseline was chosen using these same test results, and no untouched evaluation
set is involved. Phase 1 stays open until this audit is reported.

Every treatment refits the baseline on the same folds and is scored on identical held-out rows. Raw
published rows give the primary recall, and deduplicated rows are a secondary sensitivity analysis.
No label is changed.

| Treatment | Question |
| --- | --- |
| `remove_matching_signatures` | Does the model rely on near-duplicates of the held-out family? |
| `no_tcp_window` | Does the model rely on host-specific TCP fingerprints? |
| `neutralize_missing_markers`, `missingness_only` | Is a missing-value marker itself a shortcut? |
| `service_marker_only` | With the reference and column removal, what does UNSW-NB15 take from `service`? |
| `no_context_counts` | How much does UNSW-NB15 depend on its `ct_*` counts? |
| `all_controls` | What remains with signatures, TCP windows and markers all controlled? |
| `exclude_suspect_benign` | Do the possibly mislabeled CICIDS2017 benign rows change the result? |

**Before you start:** use the same data folder as the Config A run. Results go to a new folder,
and this notebook only reads the earlier Config A folder.

**If the session stops:** run everything again with the same `OUTPUT_DIR`. Finished folds are kept.

**Sending results back:** return this notebook with its outputs, plus
`aegis_artifact_results.zip` from the last cell. Neither contains any traffic rows.


## 1. Settings

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.isdir('/kaggle/working')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    default_data = '/content/drive/MyDrive/aegis/data'
    default_output = '/content/drive/MyDrive/aegis/results/config_a_artifacts'
elif IN_KAGGLE:
    default_data = '/kaggle/input'
    default_output = '/kaggle/working/results/config_a_artifacts'
else:
    default_data = '../data' if os.path.basename(os.getcwd()) == 'notebooks' else 'data'
    default_output = '../results/config_a_artifacts' if os.path.basename(os.getcwd()) == 'notebooks' else 'results/config_a_artifacts'

# Edit these if your folders differ. The AEGIS_ variables are used by the automated test.
DATA_ROOT = os.environ.get('AEGIS_DATA_ROOT', default_data)
OUTPUT_DIR = os.environ.get('AEGIS_OUTPUT_DIR', default_output)

# v2 asks for 5-10 seeds. To add seeds later, extend this list and run again with the same
# OUTPUT_DIR. Five seeds cannot reach p < 0.05 in the paired test; six is the minimum.
SEEDS = [int(s) for s in os.environ.get('AEGIS_SEEDS', '0,1,2,3,4').split(',')]

# The first value of each list is the primary operating point.
TARGET_FPRS = (0.01, 0.001)
ATTACK_PREVALENCES = (0.01, 0.001)  # Assumed attack share of traffic, not a measurement.

MAX_TRAIN_PER_FAMILY = 100_000
MIN_FAMILY_COUNT = 200

# The roadmap places cross-dataset tests in phase 5, so they are off for this gate.
RUN_CROSS_DATASET = os.environ.get('AEGIS_CROSS', '0') == '1'

print('DATA_ROOT  ', DATA_ROOT)
print('OUTPUT_DIR ', OUTPUT_DIR)
print('SEEDS      ', SEEDS)

# Config A results stay untouched. They are read once, to compare the data and reference scores.
PREVIOUS_DIR = os.environ.get('AEGIS_PREVIOUS_DIR', OUTPUT_DIR.replace('config_a_artifacts', 'config_a'))
if os.path.realpath(OUTPUT_DIR) == os.path.realpath(PREVIOUS_DIR):
    raise ValueError('OUTPUT_DIR must be a new folder, separate from the Config A results')
print('PREVIOUS   ', PREVIOUS_DIR)


## 2. Setup

In [ ]:
import importlib.util, subprocess

REQUIRED = {'numpy': 'numpy', 'pandas': 'pandas', 'yaml': 'pyyaml', 'sklearn': 'scikit-learn',
            'scipy': 'scipy', 'torch': 'torch', 'matplotlib': 'matplotlib'}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All dependencies are present')


In [ ]:
#@title Aegis source (generated from src/; do not edit here)
import sys, tempfile
from pathlib import Path

SOURCE_VERSION = '0d2bf66b0fd738fa'
FILES = {
    'aegis/__init__.py': r'''"""Aegis: adaptive cyber defense through adversarial learning and opponent modeling."""
''',
    'aegis/config.py': r'''"""Experiment configuration. Unknown YAML keys raise an error instead of being ignored."""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import yaml


@dataclass(frozen=True)
class DatasetConfig:
    kind: str  # cicids2017, unsw_nb15 or synthetic
    path: str | None = None
    name: str | None = None
    seed: int = 0
    drop_columns: tuple[str, ...] = ()


@dataclass(frozen=True)
class RandomForestConfig:
    n_estimators: int = 100
    max_depth: int | None = 20
    min_samples_leaf: int = 2
    n_jobs: int = 4


@dataclass(frozen=True)
class AutoencoderConfig:
    hidden: tuple[int, ...] = (64, 32)
    latent: int = 16
    epochs: int = 30
    batch_size: int = 1024
    learning_rate: float = 1e-3
    patience: int = 4
    device: str = "cpu"


@dataclass(frozen=True)
class BaselineConfig:
    datasets: tuple[DatasetConfig, ...]
    seeds: tuple[int, ...] = (0, 1, 2, 3, 4)
    cross_dataset: bool = False  # The roadmap places these tests in phase 5.
    val_fraction: float = 0.2
    test_fraction: float = 0.2
    max_train_per_family: int | None = 100_000
    min_family_count: int = 200
    target_fprs: tuple[float, ...] = (0.01, 0.001)
    attack_prevalences: tuple[float, ...] = (0.01, 0.001)
    bootstrap_samples: int = 10_000
    rf: RandomForestConfig = field(default_factory=RandomForestConfig)
    autoencoder: AutoencoderConfig = field(default_factory=AutoencoderConfig)


def load_config(path: str | Path) -> BaselineConfig:
    raw = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
    datasets = tuple(
        DatasetConfig(**{**d, "drop_columns": tuple(d.get("drop_columns", ()))})
        for d in raw.pop("datasets")
    )
    rf = RandomForestConfig(**raw.pop("rf", {}))
    ae_raw = raw.pop("autoencoder", {})
    if "hidden" in ae_raw:
        ae_raw["hidden"] = tuple(ae_raw["hidden"])
    autoencoder = AutoencoderConfig(**ae_raw)
    for key in ("seeds", "target_fprs", "attack_prevalences"):
        if key in raw:
            raw[key] = tuple(raw[key])
    return BaselineConfig(datasets=datasets, rf=rf, autoencoder=autoencoder, **raw)
''',
    'aegis/data/__init__.py': r'''''',
    'aegis/data/dataset.py': r'''"""The dataset form every loader produces, plus duplicate removal."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

BENIGN = "benign"


@dataclass
class Dataset:
    """Flow features with one attack-family name per row (``BENIGN`` for normal traffic)."""

    name: str
    X: pd.DataFrame
    family: np.ndarray
    categorical: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        self.X = self.X.reset_index(drop=True)
        self.family = np.asarray(self.family, dtype=object)
        if len(self.X) != len(self.family):
            raise ValueError(f"{self.name}: {len(self.X)} rows but {len(self.family)} labels")

    @property
    def y(self) -> np.ndarray:
        return (self.family != BENIGN).astype(np.int8)

    @property
    def numeric(self) -> tuple[str, ...]:
        return tuple(c for c in self.X.columns if c not in self.categorical)

    def family_counts(self) -> dict[str, int]:
        names, counts = np.unique(self.family.astype(str), return_counts=True)
        return {str(n): int(c) for n, c in zip(names, counts)}


def column_report(ds: Dataset) -> pd.DataFrame:
    """Show per-column data problems, for checking a new download before training."""
    rows = []
    for column in ds.X.columns:
        values = ds.X[column]
        numeric = column not in ds.categorical
        rows.append(
            {
                "column": column,
                "missing_share": float(values.isna().mean()),
                "negative_share": float((values < 0).mean()) if numeric else 0.0,
                "constant": values.nunique(dropna=True) <= 1,
            }
        )
    return pd.DataFrame(rows)


def remove_duplicates(ds: Dataset) -> tuple[Dataset, dict[str, int]]:
    """Drop repeated rows and rows whose features appear under more than one family.

    Without this, a copy of a test flow can sit in the training set, and a flow from
    a held-out family can reach training under another label. It runs before any split
    and fits no statistics, so it cannot leak test information into a model.
    """
    keys = pd.DataFrame(
        {
            "hash": pd.util.hash_pandas_object(ds.X, index=False).to_numpy(),
            "family": ds.family.astype(str),
        }
    )
    conflicting = keys.groupby("hash")["family"].transform("nunique").to_numpy() > 1
    repeated = keys.duplicated("hash").to_numpy()
    keep = ~conflicting & ~repeated
    stats = {
        "rows_before": int(len(keys)),
        "rows_conflicting_labels": int(conflicting.sum()),
        "rows_after": int(keep.sum()),
    }
    cleaned = Dataset(ds.name, ds.X.loc[keep], ds.family[keep], ds.categorical)
    return cleaned, stats
''',
    'aegis/data/loaders.py': r'''"""Loaders for CICIDS2017 and UNSW-NB15, a shared feature view, and generated data.

The real loaders expect the published CSV files and raise an error on labels or
columns they do not recognize, so a different dataset release cannot be misread.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

from aegis.config import DatasetConfig
from aegis.data.dataset import BENIGN, Dataset

# DDoS is grouped with DoS, so holding out "dos" removes both. If DDoS stayed in training,
# DoS would not be an unseen family.
CICIDS_FAMILIES = {
    "benign": BENIGN,
    "bot": "bot",
    "ddos": "dos",
    "dos goldeneye": "dos",
    "dos hulk": "dos",
    "dos slowhttptest": "dos",
    "dos slowloris": "dos",
    "ftp-patator": "brute_force",
    "ssh-patator": "brute_force",
    "heartbleed": "heartbleed",
    "infiltration": "infiltration",
    "portscan": "portscan",
}
# Columns that identify a host, a time or a repeated field rather than behavior.
CICIDS_IDENTIFIERS = (
    "Flow ID", "Source IP", "Src IP", "Source Port", "Src Port",
    "Destination IP", "Dst IP", "Timestamp", "Fwd Header Length.1",
)

UNSW_FAMILIES = {
    "normal": BENIGN,
    "analysis": "analysis",
    "backdoor": "backdoor",
    "backdoors": "backdoor",
    "dos": "dos",
    "exploits": "exploits",
    "fuzzers": "fuzzers",
    "generic": "generic",
    "reconnaissance": "reconnaissance",
    "shellcode": "shellcode",
    "worms": "worms",
}
UNSW_CATEGORICAL = ("proto", "service", "state")
UNSW_IDENTIFIERS = ("id", "srcip", "sport", "dstip", "dsport", "stime", "ltime")

# Shared name -> (source column, multiplier into the shared unit).
# Sizes are similar rather than identical: CICFlowMeter and Argus/Bro do not count
# packet bytes the same way, so cross-dataset results carry that caveat.
SHARED_FEATURES = {
    "cicids2017": {
        "duration_s": ("Flow Duration", 1e-6),
        "src_packets": ("Total Fwd Packets", 1.0),
        "dst_packets": ("Total Backward Packets", 1.0),
        "src_bytes": ("Total Length of Fwd Packets", 1.0),
        "dst_bytes": ("Total Length of Bwd Packets", 1.0),
        "src_packet_mean": ("Fwd Packet Length Mean", 1.0),
        "dst_packet_mean": ("Bwd Packet Length Mean", 1.0),
        "src_interarrival_ms": ("Fwd IAT Mean", 1e-3),
        "dst_interarrival_ms": ("Bwd IAT Mean", 1e-3),
    },
    "unsw_nb15": {
        "duration_s": ("dur", 1.0),
        "src_packets": ("spkts", 1.0),
        "dst_packets": ("dpkts", 1.0),
        "src_bytes": ("sbytes", 1.0),
        "dst_bytes": ("dbytes", 1.0),
        "src_packet_mean": ("smean", 1.0),
        "dst_packet_mean": ("dmean", 1.0),
        "src_interarrival_ms": ("sinpkt", 1.0),
        "dst_interarrival_ms": ("dinpkt", 1.0),
    },
}


SOURCE_PATTERNS = {"cicids2017": "*.csv", "unsw_nb15": "UNSW_NB15_*-set.csv"}


def _csv_files(directory: str | Path, pattern: str = "*.csv") -> list[Path]:
    files = sorted(Path(directory).glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {directory}. See README.md.")
    return files


def _to_float32(frame: pd.DataFrame) -> pd.DataFrame:
    """Parse numbers compactly; infinities and unparseable values become NaN for imputation."""
    out = {}
    for column in frame.columns:
        values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=np.float64, copy=True)
        values[~np.isfinite(values)] = np.nan
        out[column] = np.clip(values, -1e30, 1e30).astype(np.float32)
    return pd.DataFrame(out)


def dataset_files(config: DatasetConfig) -> list[Path]:
    """Return the source files a real dataset loads from; generated data has none."""
    if config.kind == "synthetic":
        return []
    if config.path is None:
        raise ValueError(f"Dataset {config.kind} needs a path")
    if config.kind not in SOURCE_PATTERNS:
        raise ValueError(f"Unknown dataset kind: {config.kind}")
    return _csv_files(config.path, SOURCE_PATTERNS[config.kind])


def _cicids_family(raw_label: str) -> str:
    label = str(raw_label).strip().lower()
    # Web attack labels contain a dash that some releases garble, so match the prefix.
    if label.startswith("web attack"):
        return "web_attack"
    if label not in CICIDS_FAMILIES:
        raise ValueError(f"Unknown CICIDS2017 label: {raw_label!r}")
    return CICIDS_FAMILIES[label]


def load_cicids2017(directory: str | Path, drop_columns: Iterable[str] = ()) -> Dataset:
    """Load the CICIDS2017 ``MachineLearningCVE`` CSV files."""
    drop = set(CICIDS_IDENTIFIERS) | set(drop_columns)
    features, families = [], []
    for path in _csv_files(directory, SOURCE_PATTERNS["cicids2017"]):
        frame = pd.read_csv(path, encoding="latin-1", low_memory=False)
        frame.columns = frame.columns.str.strip()
        if "Label" not in frame.columns:
            raise ValueError(f"{path} has no Label column")
        frame = frame.dropna(subset=["Label"])
        families.append(frame["Label"].map(_cicids_family).to_numpy(dtype=object))
        features.append(_to_float32(frame.drop(columns=["Label", *(drop & set(frame.columns))])))
    if any(list(f.columns) != list(features[0].columns) for f in features):
        raise ValueError("CICIDS2017 files do not share the same columns")
    return Dataset("cicids2017", pd.concat(features, ignore_index=True), np.concatenate(families))


def load_unsw_nb15(directory: str | Path, drop_columns: Iterable[str] = ()) -> Dataset:
    """Load both published partition files; Aegis makes its own splits from the combined rows."""
    drop = set(UNSW_IDENTIFIERS) | set(drop_columns)
    frame = pd.concat(
        [pd.read_csv(p, low_memory=False) for p in _csv_files(directory, SOURCE_PATTERNS["unsw_nb15"])],
        ignore_index=True,
    )
    frame.columns = frame.columns.str.strip().str.lower()
    missing = {"attack_cat", "label"} - set(frame.columns)
    if missing:
        raise ValueError(f"UNSW-NB15 files are missing columns: {sorted(missing)}")

    category = frame["attack_cat"].fillna("normal").astype(str).str.strip().str.lower()
    unknown = sorted(set(category) - set(UNSW_FAMILIES))
    if unknown:
        raise ValueError(f"Unknown UNSW-NB15 attack categories: {unknown}")
    family = category.map(UNSW_FAMILIES).to_numpy(dtype=object)
    if np.any((family != BENIGN) != (frame["label"].to_numpy() == 1)):
        raise ValueError("UNSW-NB15 attack_cat and label columns disagree")

    frame = frame.drop(columns=["attack_cat", "label", *(drop & set(frame.columns))])
    categorical = tuple(c for c in UNSW_CATEGORICAL if c in frame.columns)
    numeric = _to_float32(frame.drop(columns=list(categorical)))
    text = frame[list(categorical)].fillna("missing").astype(str)
    return Dataset("unsw_nb15", pd.concat([numeric, text], axis=1), family, categorical)


def generate_synthetic(name: str = "synthetic", seed: int = 0, n_benign: int = 6000) -> Dataset:
    """Generate separable clusters for tests and smoke runs. Not a model of real traffic."""
    rng = np.random.default_rng(seed)
    n_features = 12
    # Fixed across seeds, so generated datasets share a family structure and can test transfer.
    profiles = {
        "scan": (0, 1, 2),
        "flood": (3, 4, 5),
        "exfil": (6, 7, 8),
        "beacon": (9, 10, 11),
    }
    rows = [rng.lognormal(0.0, 0.4, size=(n_benign, n_features))]
    families = [np.full(n_benign, BENIGN, dtype=object)]
    for family, dims in profiles.items():
        count = n_benign // 6
        block = rng.lognormal(0.0, 0.4, size=(count, n_features))
        block[:, list(dims)] *= 30.0
        rows.append(block)
        families.append(np.full(count, family, dtype=object))
    X = pd.DataFrame(np.vstack(rows).astype(np.float32),
                     columns=[f"feature_{i}" for i in range(n_features)])
    family = np.concatenate(families)
    order = rng.permutation(len(X))
    return Dataset(name, X.iloc[order], family[order])


def locate_data(root: str | Path) -> dict[str, Path]:
    """Find each dataset's folder under ``root``, and reject incomplete copies.

    Copies of CICIDS2017 often also include ``TrafficLabelling``, whose files have the
    same names but different columns, so a ``MachineLearning`` folder is preferred.
    """
    root = Path(root)
    found: dict[str, Path] = {}

    cicids = {p.parent for p in root.rglob("*.pcap_ISCX.csv")}
    preferred = {d for d in cicids if "machinelearning" in str(d).lower()}
    candidates = preferred or cicids
    if len(candidates) > 1:
        raise ValueError(f"Several CICIDS2017 folders under {root}: {sorted(map(str, candidates))}")
    if candidates:
        folder = next(iter(candidates))
        count = len(list(folder.glob("*.pcap_ISCX.csv")))
        if count != 8:
            raise ValueError(f"{folder} has {count} CICIDS2017 CSV files; expected 8")
        found["cicids2017"] = folder

    unsw = {p.parent for p in root.rglob("UNSW_NB15_*-set.csv")}
    if len(unsw) > 1:
        raise ValueError(f"Several UNSW-NB15 folders under {root}: {sorted(map(str, unsw))}")
    if unsw:
        folder = next(iter(unsw))
        for name in ("UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"):
            if not (folder / name).exists():
                raise ValueError(f"{folder} is missing {name}")
        found["unsw_nb15"] = folder
    return found


def load_dataset(config: DatasetConfig) -> Dataset:
    if config.kind == "synthetic":
        return generate_synthetic(config.name or "synthetic", config.seed)
    if config.path is None:
        raise ValueError(f"Dataset {config.kind} needs a path")
    if config.kind == "cicids2017":
        return load_cicids2017(config.path, config.drop_columns)
    if config.kind == "unsw_nb15":
        return load_unsw_nb15(config.path, config.drop_columns)
    raise ValueError(f"Unknown dataset kind: {config.kind}")


def project_to_shared(ds: Dataset) -> Dataset:
    """Return the features two datasets have in common, in shared names and units."""
    if ds.name.startswith("synthetic"):
        return ds  # Generated datasets already share one schema.
    mapping = SHARED_FEATURES[ds.name]
    missing = [column for column, _ in mapping.values() if column not in ds.X.columns]
    if missing:
        raise ValueError(f"{ds.name} is missing shared-feature columns: {missing}")
    shared = pd.DataFrame(
        {new: (ds.X[column] * scale).astype(np.float32) for new, (column, scale) in mapping.items()}
    )
    return Dataset(ds.name, shared, ds.family)
''',
    'aegis/data/splits.py': r'''"""Train/validation/test partitions and the folds built from them.

Each seed makes one partition. The seen fold and every held-out fold reuse its test
rows, so the seen-versus-unseen recall of a family is compared on identical flows.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from aegis.data.dataset import BENIGN


@dataclass(frozen=True)
class Partition:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray


@dataclass(frozen=True)
class Fold:
    name: str
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray
    held_out: str | None = None


def make_partition(
    family: np.ndarray, seed: int, val_fraction: float = 0.2, test_fraction: float = 0.2
) -> Partition:
    """Split within each family so that every family keeps its share of each part."""
    if val_fraction <= 0 or test_fraction <= 0 or val_fraction + test_fraction >= 1:
        raise ValueError("val_fraction and test_fraction must be positive and sum to less than 1")
    rng = np.random.default_rng(seed)
    labels = family.astype(str)
    train, val, test = [], [], []
    for name in np.unique(labels):
        rows = rng.permutation(np.flatnonzero(labels == name))
        n_test = int(round(len(rows) * test_fraction))
        n_val = int(round(len(rows) * val_fraction))
        test.append(rows[:n_test])
        val.append(rows[n_test : n_test + n_val])
        train.append(rows[n_test + n_val :])
    return Partition(*(np.sort(np.concatenate(part)) for part in (train, val, test)))


def eligible_families(family: np.ndarray, min_count: int) -> list[str]:
    """Attack families with enough rows for a held-out result to mean something."""
    names, counts = np.unique(family.astype(str), return_counts=True)
    return [str(n) for n, c in zip(names, counts) if n != BENIGN and c >= min_count]


def cap_per_family(
    rows: np.ndarray, family: np.ndarray, cap: int | None, seed: int
) -> np.ndarray:
    """Limit each family's rows to keep training within memory. Applied to training rows only."""
    if cap is None:
        return rows
    rng = np.random.default_rng(seed)
    labels = family[rows].astype(str)
    kept = []
    for name in np.unique(labels):
        members = rows[labels == name]
        kept.append(rng.choice(members, size=cap, replace=False) if len(members) > cap else members)
    return np.sort(np.concatenate(kept))


def seen_fold(
    partition: Partition, family: np.ndarray, cap: int | None, seed: int
) -> Fold:
    """Train on every family, which gives each family's recall when it is known."""
    train = cap_per_family(partition.train, family, cap, seed)
    return Fold("seen", train, partition.val, partition.test)


def holdout_fold(
    partition: Partition, family: np.ndarray, held_out: str, cap: int | None, seed: int
) -> Fold:
    """Remove one family from training and validation; test on it and benign traffic."""
    labels = family.astype(str)
    train = partition.train[labels[partition.train] != held_out]
    val = partition.val[labels[partition.val] != held_out]
    tested = np.isin(labels[partition.test], [BENIGN, held_out])
    return Fold(
        f"holdout:{held_out}",
        cap_per_family(train, family, cap, seed),
        val,
        partition.test[tested],
        held_out,
    )
''',
    'aegis/data/suspect_benign.py': r'''"""CICIDS2017 benign rows that may be mislabeled attack replies. No label is changed here.

These are the cleaned benign rows sent from the victim 192.168.10.50 to the attacker 172.16.0.1,
99.1% of them from port 80. The reasoning is an inference from capture context: on the raw rows,
97.6% have the same 5-tuple reversed as an attack flow that day and 99.96% fall inside that day's
attack window. The features the models use have no IP columns, so the rows are listed by their
position in the cleaned dataset, built from the TrafficLabelling release of the same flows.
"""

from __future__ import annotations

import base64
import hashlib
import zlib

import numpy as np

from aegis.data.dataset import BENIGN, Dataset

# Cleaned CICIDS2017 row count with Destination Port dropped, which these positions depend on.
CLEANED_ROWS = 2199163
EXPECTED_ROWS = 38045
_CHECK_COLUMNS = ['Flow Duration', 'Total Fwd Packets', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward']
_CHECK_DIGEST = "94b4da6bb327bc0a6640437532b400e2d24533373b1a86750610981ac775945a"
_POSITION_GAPS = (
    "eNrtnXmcZmV1599737XqrX3fq7q7uqvX6ip6p5veG2gauqFpoGm2SAs0mwKySJRNWjTKxA2JRiRGQYLykSQiCckEY1yi"
    "0RAnn4lxGTNmMjOJiWYSMxPjTMLM83C/J/Xz+lZLZpJxBs8fv7r3vcuznOec85znnPPc2n2oUEgKDofjnxsnTToNHA6H"
    "w+FwOBwOh8PhcDgcDofD8f8Wnhx3GjgcDofD4XA4HA6Hw+FwOBwOh8PxYvCmMaeBw+FwOBwOh8PhcDgcDofD4XD8KFB1"
    "/6zD4XA4HA6Hw+FwOBwOh8PhcDgcjgj/TorD4XA4HA6Hw+FwOBwOh8PxY4+PDzsNHA6Hw+FwOBwOh8PhcDgcjn9p7Bh1"
    "GjgcDofD4XA4HA6Hw+FwOBz/J/jmoNPA4XA4HA6Hw+FwOBwOh8PhcDgchcKE7412OBwOh8PhcDgcDofD4XA4HA7HSwiH"
    "BpwGDofD4XA4HA6Hw+FwOBwOh8PhcDgcDofD4Xjp4o1DTgOHw+FwOBwOh8PhcDgcDofD4XA4HA6HwzE/vuD/k8jhcDgc"
    "DofD4XA4HA6Hw+FwOBwOh+Mlh9K408DhcDgcDofD4XA4HA6Hw+FwOBwOh8Pxz4sZ34fkcDgcDofD4XA4HA6Hw+FwOBwO"
    "h8PhcDgcDse/GL484jRwOBwOh8PhcDgcDofD4XA4HA6Hw+FwOByO/xs4Puw0cDgcDofD4XA4HA6Hw+FwOBwOh8PhcLy0"
    "8dWFTgOHw+FwOBwOh8PhcDgcDofD4XA4HA6Hw/GjxatGnQYOh8PhcDgcDofD4XA4HA6Hw+FwOBwOx3b/BqbD4XA4HA6H"
    "w+FwOBwOh8PhcDgcDofD4fgxw21Vp4HD4Xjp4bVXFwoPvaZQOL67UCiH35sC0oCpgLN45ifChfZw/HN+nxRwbkA17r0O"
    "eG+4OBiOFwScc232TGfAjTxfCShJnfF8acB4QBHY70UB3QE3BBwPWJ9k9bQFPFHMzpcHvJk6UhDLKNOmNwXcHDARsJr7"
    "dwTUpA1Frmu7yhy3BTwcMJ3OtbeZ51cEfD3g1oCbAi6lztjHL3Kc5vm9AU0BY7Gt4eVyX1becOjM+nDso74aZcT+DFHP"
    "DdKmkvSzRpkpdVnb98R6w8kCfi/jGMfklCQrY5jy9gjN2jhvpz1rAkYDeni/nTrbOI/X+iknsE3hQJKd38C11oDz6M+r"
    "A7p451Tul+njdUl2Pdb9s8uz8Y+/PxmwjrYdoo9fCxjhfqTrviRrZ3z3cKwnyZ6PY76B51ql7g6QQLsnoccHhB8T2pzK"
    "O2OU3wU9brssfeHewoBvtBQKvzpULXwivLyjN+P/fyinhb6BQuFI6ORbQ8PvCIT+XDiWgoztDPcHIn8HYv59KOT3cnLY"
    "VZs7X04/9X5r+HMOYzHAcVLGsJ9+RL4KTSvUA3oZz6nw7qegzTA0qPLMNo5FaDxO3+PvtfEYCDKcZLI0zHgYLUdoZx2Z"
    "NX6MY7kq4OPxPLz/5SRrbzdtaqKNJvst0PkQ5TVB7xHkrZV2lXh/in7GMpcwTl8OuCrJ3u/knaX0s0NkOB7fQ/lt0vcF"
    "PHt6wO2hnGsC7kqysqJcDCZZORug9U+F3zuT7P0/DNhdyHhgJXVX0WUpv9czrjNpphdjPYfSTFfshcfiWERabw/HHQF/"
    "AD4drl1G3/8Q3Xck4C3IU6TNSChropj1aTvXl9Ce8wPOCHg+zfpwJByHkkwPxDG+GJncDk0j3R8MuAeaPRvK3RQId0s4"
    "/27A5+jfxaGMv4CeW6FzCm3Lol9NrpajWypgnHsTvDvO2M1As4+jS+O1u0NBXwj1vTfqM8ZkhDbfl2bjsD3c/7OAvw7n"
    "F4XjPsZ4iDGL7x0K5bw79OcV4Xwj7b6aeu6P78HnUbffBs92UM5F4d3xUO4X02wsrkYndiXZ/UvD8Tuh7I/EOTP2Ocnk"
    "IOqZu8P5zwVsCu/OhuOOUNY3wvXvRRqE330Bx8L5lUkmr3HcP4mO/PXxjJcnoNdjAXdB4zrz9PGo09JsvJaKPvtYkuno"
    "teitK5mrH2I8/gQ9MphmPLoZvvoSNNsPrZsYh5voc6zn3qhL04yOp/LM25OMv4s5HjA5t3mnItcrubm4E4zSvwr8kIhu"
    "qvFMScop5sqzOVLvWR1LKGOUfsZnT4Z28Z3FAR8OD/5q5IMk44OlyVzdxquRXlvTTC/sgR8jLd5Tzt5Zjx4owmMJeiLy"
    "yy/DM9uRtWNctzl8Jvx5tjuzuy4M+HZo4F0fLhQOhkpfUcrG92XhmfeF42/FesP5M1EPplmf/jv8G3XTOch5nDveHu2h"
    "NOtzHPcnwvFxaHBTKCNUUfgW/flXtCXywCUBfx9weeT9YmZTLQ83nw/4bLQ5QgGT4fwq3jHZjfS4DlvkNwJx3x1unhft"
    "umLGx1fMZHLyGHrvr0PH7gkM2x8avyE898EgbE8FYsS2LAjvHKTtXwnnpXD/l9JsTv2fgUla02yMN4WKV0c7ITDlvvDc"
    "lnDxkcCA+8IzPxP7HZTU29qzPp0f3lke8O1KxrNRH3+KPqwpZrJcsvkxPHdLJdN7NjdH/ffySIswUO9MMr6JfXlXePaP"
    "w7MXtWe27OXdWfu/FhTSJ0IfV4frB8L9x4IwdYZ31odrK8LvBaER/yFgc3j2E6H9hwK+Gc5/P7T/eHjmmSAE28L9xdEu"
    "D9f/Wyj710IDdwcinhP69XOhjP2h/0l4b11zRtPHwvm9s2FMA3M9G4z8vwxlfO+SE68NIn3j8dhMpvsnxRaNsv5VdKXa"
    "zv9oz3CMdvJX2jI5O4yuWoaefTN2+eLw4CuRh6hTHsC2vBeaL6DuLnS56eQ+ZKmJ54ZFVw1RzyXYQF2iVyKtWymnG5lo"
    "R6bX8uwsdugE7yxi/vU1448e6Y9BH5v/N2gyLmu0Knxt67XW3FqnQ56N892Z5ez5IbMZk2zeGKUtw8x1A8jBAnS1yV+c"
    "E38Du/84a/9r8BOMUN8CkeFm7PYpymyj3C50ybuSbE25ABmNMriFssZ4v536V6EfjjIPjjP31ChvnDLWUPed6KFoF78e"
    "xHnkPOb9dtrQyfvNvG/r/Gba3En9HbS9D71RbMroencxK6sFP8Q2yhvDRra11jUyVo7/f7GcddaU+CDi2P9599wzR+Gx"
    "IfjZ/CyPcozr3X3Icx981QE/DzMPdWEX91DHOvhuGHv9CO/fI+exbbuww34FH8he6o3+oZtLmaxPU+/flrI1V5SLzwc+"
    "fluayVML9uEjzJW/h+xG2diJ/VukXdHu+16aydp7kJ0ovwcDPpNmZRxEJ9XpW6fo92bKUp3YJ9eGkZuiXOvBd1eT9Yf5"
    "q+y5AcbI7i+FNv8aXXaA/kT798Yk8+u1U1+KLvgU66kJsYn68I0upM6irIdvE79JKu1tkf6m0OAOaPU4x/tZmzRJXWUp"
    "39ZafxV+vDmZ89sdh5+iH+7TtGMt/d+aZON9I3ppCWu7F9ZSSaa3p/j9P+hTM3RpR3e2sn5eTDvaoEc3dCjDT1Wem2E9"
    "sQJeeS3rB/NBxDJ+nTkn6uTbxWcU9edH8U+fwbywlr53UEcJey8+e5/4pFYjH2X6UUP/38K7FXy65q+5NuBs3r+GuirU"
    "YWN2MvUt4RhpPJnM+ZZtXdpFfT+VzPl6u8RebqF/r4ImM8wLzZSdQFMb5xW8E/1H3+D6JG0v4+MYZqzj2vMqfj/JXHUE"
    "O/x0rvfyXhdYJOvpVfBADzToZIyLzKGHmcvugh67adshWX+fTZtPEhukzjjezvVJsenNb1uEz0Yo71yxX3pZK59Ce/ZA"
    "szPwZ5TRQwOct9OHa6FlM+N5Ku9vgefMrjmF+o02l6Mnf5I+me3yCuhzGuVOoQNOFhntxM+0h1hBD3ZTp+i102Q9cyv8"
    "WhQ/7RTzRRPPL4TOCfL7s/Q3jmlcR11JG1+G3O3hmHL/ffRtj+jTL1JuLON6ykug4dfRjyvgj2gHPg9/1Cm3C99oE7L1"
    "OuzISegQee3sJFuTjXC9jzHohp5twl+2TtzN+3fKuEa/2gbqPSh+/49C69vE91OBdhX0vI3RColfnSnrQPPXHkKHH4LW"
    "NebZRGzmFuS1j76aHbxU1qGmD7oZg6OMjdmQVWhjvugJ6NovdF3F+Yisra3+sqwVTG4trtRLuUs49lFft8wfvdDO9HaK"
    "nliDLJShcZV6W+DdMXRJG/0sw8eml3fwbDt8VuF8jDpmREaqYgNN0K4z4e02iY+1cm62zzZoup05aAZaxvV9Bd2ylbp3"
    "UG4Pv01vL8Q2WsYxlvNOaL6dOssSO0plTE12bqLfzbTffJs9EqdJ51m7TonvYRI7aYD56zu0uwLdjtHOoox3Kr5Tk59O"
    "2jyBz20MOTqfNvdjG16SZHwbdU5Hkuk98+10Izd14nkW43mN1F3K9atVaFOkPyvRrxXKPQk6R9o+I/7dPfiEEvitLDG4"
    "i5kv4vtv4PoKxujv0LdV5ON2dNTlzE9bkMcVjO1iZHAAeW6CLhPwt8Vhq8jGdvo2Cp7hOaP5sPjGn+b6atq2mvaPIVMb"
    "0GM1kZk0N44WQ7mC+Wa18SDr0/8i9sBKxmQjYxr56I1Jdu+3oW2bxKwGoL2OX6TVL+R40nzow9De7Nw6/DOObF/fwCdT"
    "Yg4pQpNHsclehX/X7h8AFfytlZzPoyg+xCnmiTh2zxEjPoreb+W99cxtTfDzXni/R2y3XnTUhbT/HHigBt076etubIkm"
    "fpuuXIP87cOuOVd063p0wLn4L/NrQ7NPD+IrtnlzI3JWRH9NMH9dwHuPS/zJ1kODlDVJ/ePctzXRGPTuQ0dbLkIi+QhG"
    "38gXvwltFufmrNi+y6BrSWL4kY9+kbb9BHXaeK+Teel62rYXOs5ytDXZTfCwxXwm4c8bKaefe90S97E57pj0oYkxm8Re"
    "PwR9N4sNM4mcj1HnKbK2KMKL6Q/xN+aPa9BHPeiXu1jvLJJ1aRN+7rzuN5m/DnotZ8y7sLWWwRtF4ZUaKIp+3Ma7v4As"
    "VODFU9F7qfBxInbmqdS1j3u2hv8ucderpK0dzKN16m2C7yx2ewQ6TjNmFsNdiBytZEx1Hjad2o7MWFz/Yeq7lrHqQheu"
    "ROZnsaP35HSF5vDE3+ck2Ri8GnnplnyJmxm7CrxyPecW31gr9lad/h6lvU2MzyxlrhS9ZWs0s6t3MTYnwyPvgh7N6MS/"
    "g37jkjtjtuUzzHmHsIWbiRXXkozGFbGfUvq0Dd5bSTteznGR5G1Mwxe99G0NPNCX8w/bOmcn6+R+5Ph55qSV8OgM7zQJ"
    "/cuyrllYnJuvrK2/i875j+JTtjVZJ78nKNPit8slLjqGHFSooyyyVkGv27Xr6Pe5zMlV3hlFHwxBg3JOvmMd/z7gg+i3"
    "7cSAt9H30QYyXUGv/Bn3n+C+8Yv5wQeR7Vje+3mmKj6mg5J3VZS8ms20c734kS2WPSlrBo17l/CzTCOPNo8tR2eeTJ3j"
    "kuM2xBxtOV/76de9oq8Trs2CIZ5bwfkbsR1aaMNebL/zaNtZ8JHq2SZ48k50wevob4+sx+zZdfBKXeyxAeHhsuiFIn3b"
    "JHkKW2iT+dwW8+xV0DKBJpv4bX69BYyL2aMXMa59rFn6eM76Ynap2cTPU94a5Oo85O8o8q/rvKLw1/kSkxwlH6goPtMm"
    "8S/W5b0/RQ/sYJwXwj9T0PNM7KFEdEqVsnfhQ7wG/WJxkX6eaYYWD9KfbdBnFLosln7PIH/LyBVcKOvPAca4Dg0WCe3M"
    "R2W+J4vhtHAclvYWZe1/SNb4KyVXpgX5vymZ01cX07czxOe2SNYmicyzs/SxmLPdzRd5nuRv3kj9DzM+5jc5ILqjlNMj"
    "+Th6j8jxRvzuG2QtexpytFria8OMwSL6tVDm/6rYCWVoe57cr6N/dqGn+uh7q/gxTNebb8fsklUyFmZb6PycsrbexTi8"
    "lfKNX08R+33YcqvgmQF4SOV6ATqnzvM12lyG1lPoyDeQq3UE2j2M/HcLz9saYMh8OknWzo3I+kPIZQs82CxxnqKU0ZYb"
    "v7UyjvvhIfPJLBU76NUSC2hj/T9Lf3Zh8yyHJinPvonzvTIH94IP0edZ7Mp28Z8M0p5Hef8AtFhOfk/0/z8g84/l/3yA"
    "5ywH9SQZ14rozBF5bzP270LkcSu0XyM+zEpO1yXwzkn4Mi8UX+oLce4ki5FUJZbTznFjg3n5dHTAALw+w/nfoLu2sZ67"
    "ivFvZ3zWiK5Pczp2IJc7XqPu36a+acp4P3PsLLxXFxvpk4yd5akOEfeo05fjjMl/kljBKPV2yJpqN/WfJjEJk5OFsl6o"
    "yTjtRRZH4Ie9zAuD4m+riL1vucWWZ3wYuh2kDRXodK3wxTjlDXB8TQP5mA999Ofl+KvbibltpfwSdS6H7l3IjL3/O+jf"
    "W2TNGNv7EZGjuuTxWV7qFDwxJbohEV9WB/RQX2YZvjLf/9VCl4epZzf24KjwfFn6ug07zXIiRhnHbTK/RD3wNsrqZA0x"
    "JvnLuoZIc+ujkvjca6IrTO6N3ivEJ1xn7b9Y+C3W92/ETzZFbGmf2LA6jtexHtrMuF0OjdZDg83onyXwiNo+Rp/PiY5Z"
    "TJ07sM2nyZE+S+zQTuH/TvFdDVLPBcI/h7CtThZf9jT1Wfxno/BQF365utg/l8lc3ix7TiYlb3AHx7bcfKj5cofx/+4n"
    "dnOQtu2XdXWZsZqm/Xc3oNcCdOwv04fY/h3kBn8O3ipyfank4yTi0zY/puncTpkjr5Z4Uj862uzoJuTE7CHL/b4N/ZqK"
    "LbZQdOgW+Hsz/Y5rxHfncoatn/FkC33fhH1wHPujXeaDUfpWlfljTGIdLfS7BdvDaHC52HXNorOKEm9eN08+WpH2nAx9"
    "a/DlZyjrYeaVww328Oga9A7G4lroZLbwZfDSLq4vpb39snazsi4Tnl6Rm68atf2oxemwmVLG2fKVNDb1gMzvYyL/F+dy"
    "OBJZkyb4Wr4tOVUH0GdHGLd3yDtny7h/FLodg3cepG/LiMM+LX7RRPaL5H2wCbGPkuSd7ILnLN7XgazavD0u+1osBraT"
    "a++XuGET41pu0P8tudyNzblY0l7xLaWin/LjVZb11Fsppw1dsxaf6hZkt4Yc7KBsW/sPct4vvpVpZEn5uIn4h8UyZ+j/"
    "NONxNnpXc2uGeH5A9JHJ73pouRJd0Sa58Gbjb4ZvL5B1+hi2SrEBP+XjFpYzcBia1GSuXQVfl2TMkty+QPXNDqLrLMfg"
    "Idr/Dnx5JYnljudiIOq7WSlxzStzfBnl7Sn6XUSnmX2+DD1zgPZupC22v2gTPLqOMTQebJN1RKvomSrr/S7otEZs2Y9J"
    "fKI9x5vW1uOiuy0X69+yX2SasV87Ty6j0qaLd3fTvinaMsHYreT+WmhzA/bVadBmMWNZQQ6LvHMNsnslffimrI8GmH+2"
    "814/tpX17+JcPPNU5t5j8KLtX/luzg/1IZ7roe61PH8W43Uhbevh9yrO+2hXWeLQA5LDvkfygZrF/qjTt6Lk5Hagu1ch"
    "Ly2UtUnGsiTrLMtnuBUa2F6NkvCKxSLL+OFOl1zi7gZjG8v9S8kF2y85eBbT3Umbz8NHW5K49XKJ015Hn69PMjlWX8kS"
    "aLoXuWiln2vo69n0fRZZWCt67ixQI048JvPCsOiVVPrbSA5S4rGLxLbbK/toF9PWJ+lzM7JjsZTX8lyU21+DJ5uQg3rO"
    "Zjf9tFB8qV3SjmHJzbD8k1TWA5PYgJVcDGlY8kmLOb+T5Ta0NbARbP5aIPtQt4q+75U1ZJlxXi65XU38bqVPGySenIpv"
    "d1r4vpTLCW3P5YbavYuY90rM41Zui/hMxiVeb/mzH5b57pDkayeSX9OWo8GM6NtmyeOdD/tYr5XRB73w3wrq3g7P6r7U"
    "VPY4T8H3p0qcI6Gdlrfb0WB+vFrmvxJr4mPQfQqf55sZr36xm8tiM7fhf7L5epPYNOvFr9SMDioRpx+WuagRTT4oazrb"
    "J/ocfPEd7Lqi5FI3KqMJP9gyZPYI88rpuZzdB4krJLJG6mTfpK2b3yXj3jnP/LUoZyNU52mXtfsryMACkRvLR3sk922E"
    "Eu0330ufyEuz2PjK90tz+SNJg3al89j+SW6ufiW8mYrv5i3Ca7YutLzCpWL3pOj1nYyffdvhZNFpSS6/wNq9RHJpBim3"
    "WdbPCWWPIDcaq6o0oPuQ2GBLqHMbc6XlxFwkMl3GLhqWPqY5X8ZgAxq2ItedYrMNo78WSk7RMLJSlvVZeoK9TPPJyzLa"
    "1ISMLhZ/Ugs0apb8uVbZL2f+rpr4gG0ur8NjdfFpGf+s591O5NH4sNbAZ1sUeloe8cyL3LeV4uuLdX1W9s8XczZGXfRZ"
    "kbXCAHRYL89VWR+lMjful9wpi/MbD1je3GL5PsGlQjONLw6LnZ2I/tc8PIuVWYzavm3SWpj7NseJxr0m166S/JYW0cFF"
    "+HuJxNpsr4PFYgcl9mC8d67E3XppT88J8l3Opx/7kedOyj6f91ehEyxHeVzyFe9mbiqLv8zWoHvoW0l4yXz6u2UdOSj7"
    "LirQ92KJZb2Cdr2OtnSIbbxE1hPD1H0a4zORW5ckORvQeO8a6BNlKu5HXiLz0StFD98quu6CBmPbSFbG0Jn3MC5pg9iv"
    "vrORui/CZ2U+jJ9EJ/dy7WbJUTI7+W3YGJY3NC6+/vNlLWv0OKVBPDKfa6P9m6Yt8d1nxUayvJFknvnJsAr7ZlDWd2a3"
    "bEOvmk2yHH68GTtmKrfGeYS5/NOM/53i12lFX1TFt25zj9oZq8V/1sx9i1m9njb8lvjg82OtZZ0sNt8qyYe+gbbZt1Te"
    "Dt/O0qe3SG5Cni8Oiz+yReLb5qvv4FodOWiWcWsqzH1PZlryobrRB12yj0vn0laxDWvz+PmGxYZJcnHysuQpN0vej+Wz"
    "bZZyhkRnlcQeaGkQZyiK30+/kdQn678WiRNvyMXxKlJGc04n9AjdKujxiVyeSbv4mgdkrVYVX3cq7xZFBooSv32BnkmW"
    "V9lxgjmzlLu3RvROUWxysxvaiL8sk/h/i/gFrayt/F4na0azmUagWxdlGF0shqjfWio1aG++D/UT2KVm213BeGwuzH1b"
    "53clrpLPpa8y3s3wt43xfnTmVuS2G52nsbg90GdY/GnnSL6w5ZbZHLRUYjcPSIymGV7Yhey2SGzFeO3RHN1b6esE/qsR"
    "eOUW9MbSBmv0yjzrjlcyHpaT9ZTkKNle0C+h3ztEju9uEIt4N2vEdtnP0A9NKuzns3z3jWJjm82eyN7up6GVyb59V+tZ"
    "kblF4p8y+7PG3JuIzWOyvVP0c0liuFNi003JOCe585WsFS1XvBu9+irJmbqUZw/k1mrN8+TulBl7s7Fac3bFWvh0DH4r"
    "Yc8dSeZ8O/3CNxpHu0jivJZft7kw952KRPbU1Lhu+0Ftn8x6WTfMt27tQj6WSKzmZYz1ZyRPoUxMqwqfaAysLv419ZHY"
    "9/8mpR9VxrMsemeKPMKt0LOW8wdZzm4dXrbvH7ShbxfLPDZQeHHfkxgnhnKT0Nxyrg4iR2sL3/+tIfvWWg90bYXubRI7"
    "TvBNlqQ9Z8IDlt9m3yCbZazMn9kpOrgNmbZcuPOwIbobzBX6ew/5EVvE/kpz0PGvSk7YIqm/XfYBX0+5PfRrVnI7zf9S"
    "os56rm1DIhtjkhe0gjFYTblNcv9KxnRWaL9CfOy3wncmhxeKHTIGffsbrHFKufig0mc+nhkXPh2ReOug+PMG0R/bqH+z"
    "7KPqRc/sEj/iSvwJO1n/5PNhTPc+SZm23mqF/80XbXsGl6C7iuSXDZ/Ah9YkfsGa+J+qImO7ZE/qBtm3o7bTZGHuWz07"
    "6at9oy+29ynyLrfLnkubU8+ALqPSrgXUtZQyRhvkTZrv9jJkdAV0WsselWKDHAnlgWnhqUTm2prMHRr/r8r+qJQx64bW"
    "/ZTXwxhE/fxe5twKbTRbr0di6I14bQNrwlliIhaz7JE4tu1pG5C5r0ns5hZZnyzmmYWFubjKi/22zHbh06LMrVXJP22m"
    "jjtYly2CliPwa5V4odlI0zneSQs/+L1TtZMX5HItN1BfLO/fiR391h+i440PIm2+wNy/E/lrp93m63pHMpebN8j6bPU8"
    "tnhJfIUjwhu2BnsUndnbwH7XNnbkZLObeNVa4cul+DHq8MiGwty3erbSXhujIck5LUke0bGcnis28G92S77CjPgHR+Hp"
    "94mt2JSbT0qyNhiRWOT7kYPV2Lxb6N9u6XOZsbW8O9OTG7DDDjewBZR2xZwPp5Q7TyQ2UZNrp7FWL8vekEfEz3R24cV9"
    "q0fr208Ztl78xxxavot6juSktrJOGxJ/cqvMxSfi6xezfy2VvOOEfAjb/z0mdNTcG5s/+5AZ2z88JrHma8VPk+TWNebP"
    "/M0f4ueejw/n2ztXk7V+Bzab7dGyvZGd8F0F/TEie67W0v6Vsv+tizhd/QRzZVnWQmnOZlrWwGddFpvavoN1JrSyOeb6"
    "wtx3Ed7AGL1d9o3WXqSeLsE7o6zhV5ALG+v+gyRbZ1wouTGJ2Hg2dmckc9/kVjuxJn5Y/W51Kro7n6eUFv5p33QrnuCZ"
    "YoNnSpInZDy8NZmL99p3ZSwH6nHRn8V5/JqWN32V5EHulLVESdZ0JeotS17RkMSUzB68BN6owpd1yUOx+fQciY8l4vdO"
    "GtC3U3ijFfsgRUdZfNu+H6Vj1QrfNzfw25Vy+6XuKcx9P9niSjWJ9ZXhmUmx/y3OYd9Fsxz6PfS1ozD3PQirp1PWET2i"
    "X8sib7tlL0iF9YXldbYDi0vZd0eqsjc+EXu0Uc7Y1tweGNvDOpx+v+yXZA+c5Y1slfLeCT8szK0nig14rE1sszhORytz"
    "viize7uafjAendet+s3bpsL3f4+hnzG4VN7pYW6dxl6+XfJT+iX+2iR+1msY6258FS8nXm7fNXnhG4JJJmd3Fea+Hbxd"
    "+GtG7Osh4Rfbl98rNkSrjK996/cksYEnpM8T6Ls+/FXD6Dj7Xn+S8znH+/czRifRh/3i271L/PO9wmOp7NMwe9via3fK"
    "nrUuybXvllzHTvG9rpRcNIstFWUP2YDU1yE0aJY9VovE1zglOqkoY2nf0bfcuvPhrV7WjKvRR32ynlsM3TpEFus5/WA8"
    "OCNxbvsGwuHC9/8vg6RBnmXeHj8k/FkUf9I2+OWRwty3HCbE5tuJHWp9OSx7Exbw7Gxub8R8OaTr0CsVyZdcIvK8EtrU"
    "qG9PYe5bJNXCD37HelzyyceQmRbxF3zM8qr5lvOojN8sY3K/yMVioX1d9lmluf18ffy+VvKD19GXFu7fxpz0asb9aI4e"
    "VdH3ZhOukHlkiejvsvi+43j9fIP1ufpcazIn76Ev+8T30ITePkX2mdmc9hHG2XKTLf61FP+kzYdvpJ+vp//27bcjoksO"
    "stct2vtfSrL9YHHtup5v/pXpc6WBnVAXW6AsfhGzr2fwR6XCM/p+TfhMfbzmy7hc9lH1iE9pBbQ5S/aW72J8SxKv1HVz"
    "1IV/dTR9IS7wXHjo80m2Ro4656PhofbQgcdDYa9LMr54KjyzMGBjwNpqJmO/Ep75UHjmL8K1y8Nzf5TM5SQ8RU7/fdD4"
    "CmQ3fpfsvvDc35bnvuEW98Z/iN/PhPL+JjT0PyeZTv6lGK9tzv5XygPhmSfC8Ypw7/cr2bdU4/cP9oVr95WyvNv4/wzi"
    "mMXvo/9pOP9ueO7m8PyfJNlcFMfwkfh98vDu/kCUj4X3rgjn7wnH5iT7vuWeYvad/acD/iF+37+c+be3EK/4xbgXMuCn"
    "6esZoZ5vBiE6KdDlW+H6x0OdO8PvT4TrT4f3TwvX7k2y/5kT/0/A98Jzx+I3ySvZtw2jvFwV6vx6wAfCc98qZvwTv91+"
    "bZL9n4+HwvM7Y/9DeWcERrkfn9nnyR1ta8rkPPbzsfBMPQjAc6GeajjfzP+kiP+TIK634u9h/t/Bf41jGso+JfxeXsli"
    "CdfxPyOOh+vL+J767wT8TJL9P424pro5zeT6uTSja8y3eyK81B/OH0yzd+K36u8qZfNvb5rx5GfJlV+DLf3H8OpXY7+j"
    "rVPKcgAj73wn/D7E/24YqGR2V+TJr4ZnVobj10LbJovZHPO1csajf8T/W/hfP++Gsg=="
)


def suspect_benign_mask(ds: Dataset) -> np.ndarray | None:
    """Mark the suspect rows, or return None for data these positions do not describe."""
    if ds.name != "cicids2017" or len(ds.X) != CLEANED_ROWS:
        return None
    gaps = np.frombuffer(zlib.decompress(base64.b64decode(_POSITION_GAPS)), dtype="<u4")
    positions = np.cumsum(gaps.astype(np.int64))
    values = ds.X.iloc[positions][_CHECK_COLUMNS].to_numpy(np.float32).tobytes()
    if len(positions) != EXPECTED_ROWS or np.any(ds.family[positions].astype(str) != BENIGN):
        raise ValueError("The suspect benign positions do not point at benign rows in this data")
    if hashlib.sha256(values).hexdigest() != _CHECK_DIGEST:
        raise ValueError("The suspect benign positions do not match this CICIDS2017 copy")
    mask = np.zeros(len(ds.X), dtype=bool)
    mask[positions] = True
    return mask
''',
    'aegis/evaluation/__init__.py': r'''''',
    'aegis/evaluation/metrics.py': r'''"""Detection metrics at fixed false-positive rates and realistic prevalence, plus statistics."""

from __future__ import annotations

from typing import Sequence

import numpy as np
from scipy import stats
from sklearn.metrics import average_precision_score


def threshold_at_fpr(benign_scores: np.ndarray, target_fpr: float) -> float:
    """Return a threshold that flags at most ``target_fpr`` of these benign scores."""
    return float(np.quantile(benign_scores, 1 - target_fpr, method="higher"))


def prevalence_weights(y: np.ndarray, attack_prevalence: float) -> np.ndarray:
    """Weight rows so the test set behaves as if attacks had this share of traffic."""
    n_attack = int((y == 1).sum())
    n_benign = int((y == 0).sum())
    if n_attack == 0 or n_benign == 0:
        raise ValueError("Prevalence weighting needs both benign and attack rows")
    return np.where(y == 1, attack_prevalence / n_attack, (1 - attack_prevalence) / n_benign)


def precision_at_prevalence(recall: float, fpr: float, attack_prevalence: float) -> float:
    """Precision implied by a recall and FPR at this attack share (NaN if nothing is flagged)."""
    flagged = attack_prevalence * recall + (1 - attack_prevalence) * fpr
    return float(attack_prevalence * recall / flagged) if flagged > 0 else float("nan")


def evaluate_scores(
    y: np.ndarray,
    scores: np.ndarray,
    threshold: float,
    target_fpr: float,
    attack_prevalences: Sequence[float],
) -> dict[str, float]:
    """Score a test set using a threshold chosen earlier from validation benign traffic.

    ``recall_at_oracle_fpr`` picks its threshold on the test set. It cannot be deployed,
    but comparing it with ``recall`` separates ranking quality from threshold drift.
    """
    attack, benign = scores[y == 1], scores[y == 0]
    recall = float((attack > threshold).mean())
    fpr = float((benign > threshold).mean())
    oracle = threshold_at_fpr(benign, target_fpr)
    metrics = {
        "recall": recall,
        "realized_fpr": fpr,
        "recall_at_oracle_fpr": float((attack > oracle).mean()),
    }
    for prevalence in attack_prevalences:
        metrics[f"pr_auc_prev_{prevalence:g}"] = float(
            average_precision_score(y, scores, sample_weight=prevalence_weights(y, prevalence))
        )
        metrics[f"precision_prev_{prevalence:g}"] = precision_at_prevalence(recall, fpr, prevalence)
    return metrics


def bootstrap_ci(
    values: Sequence[float], samples: int = 10_000, level: float = 0.95, seed: int = 0
) -> tuple[float, float]:
    """Percentile bootstrap interval for the mean across seeds."""
    data = np.asarray(values, dtype=float)
    data = data[~np.isnan(data)]
    if data.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = rng.choice(data, size=(samples, data.size), replace=True).mean(axis=1)
    tail = (1 - level) / 2
    return float(np.quantile(means, tail)), float(np.quantile(means, 1 - tail))


def paired_comparison(
    a: Sequence[float], b: Sequence[float], samples: int = 10_000, seed: int = 0
) -> dict[str, float]:
    """Compare paired per-seed results as a minus b.

    With n pairs the smallest possible two-sided Wilcoxon p-value is 2 / 2**n, or 0.0625
    at five seeds, so a five-seed run cannot reach p < 0.05. The configuration file allows
    up to the ten seeds v2 specifies; the interval and effect size also apply at five.
    """
    x, y = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if x.shape != y.shape:
        raise ValueError("Paired comparison needs one value per seed on each side")
    keep = ~(np.isnan(x) | np.isnan(y))
    diff = x[keep] - y[keep]
    if diff.size == 0:
        nan = float("nan")
        return {"n": 0, "mean_diff": nan, "ci_low": nan, "ci_high": nan, "p_wilcoxon": nan, "cohen_dz": nan}
    low, high = bootstrap_ci(diff, samples=samples, seed=seed)
    p_value = 1.0 if np.all(diff == 0) else float(stats.wilcoxon(diff).pvalue)
    spread = diff.std(ddof=1) if diff.size > 1 else 0.0
    return {
        "n": int(diff.size),
        "mean_diff": float(diff.mean()),
        "ci_low": low,
        "ci_high": high,
        "p_wilcoxon": p_value,
        "cohen_dz": float(diff.mean() / spread) if spread > 0 else float("nan"),
    }
''',
    'aegis/experiments/__init__.py': r'''''',
    'aegis/experiments/artifacts.py': r'''"""Artifact-resistance and sensitivity audit: does held-out recall survive removing dataset shortcuts?

    python -m aegis.experiments.artifacts --config configs/baseline.yaml --out results/config_a_artifacts

This audits a baseline that was selected using these same test results, so it is not an unbiased
estimate of generalization. Every treatment refits the Random Forest baseline on the same folds as
the reference, so differences are paired by seed and family on identical held-out rows. Recall on
raw published rows is primary, and deduplicated recall is a secondary sensitivity analysis. No label
is ever changed. Results go to their own folder, and a finished fold is kept, so an interrupted run
continues where it stopped.
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from aegis.config import BaselineConfig, load_config
from aegis.data.dataset import BENIGN, Dataset, remove_duplicates
from aegis.data.loaders import load_dataset
from aegis.data.suspect_benign import suspect_benign_mask
from aegis.data.splits import Fold, eligible_families, holdout_fold, make_partition
from aegis.evaluation.metrics import evaluate_scores, paired_comparison, threshold_at_fpr
from aegis.experiments.baseline import FoldStore, Progress, _write_manifest
from aegis.models.baseline import RandomForestBaseline
from aegis.provenance import source_hash

EVALUATION_ROLE = "artifact-resistance and sensitivity audit; not an unbiased generalization estimate"
REFERENCE = "reference"
ROW_KEYS = ["dataset", "seed", "treatment", "family", "target_fpr"]

# Columns left out when comparing flows, so two flows match if only their timing differs.
CICIDS_TIMING = re.compile(r"Duration|IAT|/s|Active|Idle|Bulk Rate")
UNSW_TIMING = ("dur", "rate", "sload", "dload", "sinpkt", "dinpkt", "sjit", "djit", "tcprtt", "synack", "ackdat")
# TCP base sequence numbers are random per flow, so they would hide otherwise identical flows.
UNSW_SEQUENCE = ("stcpb", "dtcpb")

# TCP window sizes reflect the sending host's network stack as well as its behavior.
TCP_WINDOW = ("Init_Win_bytes_forward", "Init_Win_bytes_backward", "swin", "dwin")
# -1 in these columns means no value was observed.
NUMERIC_MARKER = -1
NUMERIC_MARKER_COLUMNS = ("Init_Win_bytes_forward", "Init_Win_bytes_backward")
# The publisher defines "-" as "not much used service": a catch-all category, not missing data.
# It still separates families strongly, so the treatments test what the model takes from it.
CATEGORICAL_MARKER = "-"
CATEGORICAL_MARKER_COLUMNS = ("service",)

# A treatment shows shortcut dependence if the mean of its paired per-seed recall drops is at least
# this large and the 95% interval of that mean lies entirely above zero. Set before the run.
SHORTCUT_MIN_DROP = 0.10
RULE_TREATMENTS = (
    "remove_matching_signatures", "no_tcp_window", "neutralize_missing_markers", "no_context_counts", "all_controls",
)
LABEL_SENSITIVITY = "exclude_suspect_benign"


@dataclass(frozen=True)
class Treatment:
    name: str
    drop: tuple[str, ...] = ()
    neutralize: tuple[str, ...] = ()  # Numeric marker becomes NaN, then the training median.
    remove_signatures: bool = False
    missingness_only: bool = False
    marker_only: tuple[str, ...] = ()  # The category is replaced by one flag for its marker value.
    exclude_suspect: bool = False


def signature_columns(ds: Dataset) -> list[str]:
    """Columns that define a flow's signature: everything except timing and random values."""
    if ds.name == "cicids2017":
        return [c for c in ds.X.columns if not CICIDS_TIMING.search(c)]
    excluded = {*UNSW_TIMING, *UNSW_SEQUENCE}
    return [c for c in ds.X.columns if c not in excluded]


def signatures(ds: Dataset) -> np.ndarray:
    return pd.util.hash_pandas_object(ds.X[signature_columns(ds)], index=False).to_numpy()


def missingness_frame(ds: Dataset) -> pd.DataFrame:
    """One indicator per missing-value marker, and nothing else about the flow."""
    indicators = {}
    for column in ds.numeric:
        values = ds.X[column]
        if values.isna().any():
            indicators[f"{column}_missing"] = values.isna()
        if column in NUMERIC_MARKER_COLUMNS:
            indicators[f"{column}_marker"] = values == NUMERIC_MARKER
    for column in CATEGORICAL_MARKER_COLUMNS:
        if column in ds.categorical:
            indicators[f"{column}_marker"] = ds.X[column] == CATEGORICAL_MARKER
    return pd.DataFrame(indicators, index=ds.X.index).astype(np.float32)


def treatments_for(ds: Dataset, has_suspect_benign: bool = False) -> list[Treatment]:
    """The treatments that apply to this dataset's columns."""
    columns = set(ds.X.columns)
    window = tuple(c for c in TCP_WINDOW if c in columns)
    numeric_markers = tuple(c for c in NUMERIC_MARKER_COLUMNS if c in columns)
    category_markers = tuple(
        c for c in CATEGORICAL_MARKER_COLUMNS
        if c in ds.categorical and (ds.X[c] == CATEGORICAL_MARKER).any()
    )
    context = tuple(c for c in ds.X.columns if c.startswith("ct_"))

    plan = [Treatment(REFERENCE), Treatment("remove_matching_signatures", remove_signatures=True)]
    if window:
        plan.append(Treatment("no_tcp_window", drop=window))
    if numeric_markers or category_markers:
        plan.append(Treatment("neutralize_missing_markers", drop=category_markers, neutralize=numeric_markers))
    if missingness_frame(ds).shape[1]:
        plan.append(Treatment("missingness_only", missingness_only=True))
    if context:
        plan.append(Treatment("no_context_counts", drop=context))
    if category_markers:
        # With the reference (marker kept as a category) and neutralize_missing_markers (column
        # removed), this completes the service comparison. Adding a flag beside the full category
        # would give a forest the same information as the reference, so the names are removed.
        plan.append(Treatment("service_marker_only", marker_only=category_markers))
    if has_suspect_benign:
        plan.append(Treatment(LABEL_SENSITIVITY, exclude_suspect=True))
    plan.append(
        Treatment(
            "all_controls",
            drop=(*window, *category_markers),
            neutralize=tuple(c for c in numeric_markers if c not in window),
            remove_signatures=True,
        )
    )
    return plan


def treat_frame(frame: pd.DataFrame, treatment: Treatment, indicators: pd.DataFrame) -> pd.DataFrame:
    """Apply a treatment to these rows; ``indicators`` holds the same rows' marker columns."""
    if treatment.missingness_only:
        return indicators
    if treatment.neutralize:
        frame = frame.assign(**{c: frame[c].where(frame[c] != NUMERIC_MARKER) for c in treatment.neutralize})
    if treatment.marker_only:
        flags = {f"{c}_marker": (frame[c] == CATEGORICAL_MARKER).astype(np.float32) for c in treatment.marker_only}
        frame = frame.assign(**flags).drop(columns=list(treatment.marker_only))
    return frame.drop(columns=list(treatment.drop))


def treated_features(ds: Dataset, treatment: Treatment, rows: np.ndarray, indicators: pd.DataFrame) -> pd.DataFrame:
    return treat_frame(ds.X.iloc[rows], treatment, indicators.iloc[rows])


def training_rows(fold: Fold, ds: Dataset, treatment: Treatment, flow_signatures: np.ndarray) -> np.ndarray:
    """Remove training attack rows that match the held-out family apart from timing.

    Benign rows stay, because they cannot teach the model to flag the family. Removal follows
    the training cap, so every other training row is the same as in the reference fold.
    """
    if not treatment.remove_signatures:
        return fold.train
    held_out = flow_signatures[ds.family.astype(str) == fold.held_out]
    is_attack = ds.family[fold.train].astype(str) != BENIGN
    return fold.train[~(is_attack & np.isin(flow_signatures[fold.train], held_out))]


@dataclass
class RawRows:
    """Rows as published, before cleaning: one entry per distinct flow and family, with its count."""

    X: pd.DataFrame
    family: np.ndarray
    flow: np.ndarray
    count: np.ndarray
    conflicting: np.ndarray
    indicators: pd.DataFrame | None = None
    suspect: np.ndarray | None = None


def build_raw_rows(raw: Dataset) -> RawRows:
    keys = pd.DataFrame(
        {"flow": pd.util.hash_pandas_object(raw.X, index=False).to_numpy(), "family": raw.family.astype(str)}
    )
    groups = keys.reset_index().groupby(["flow", "family"], sort=False)["index"].agg(["first", "size"]).reset_index()
    conflicting = groups.groupby("flow")["family"].transform("nunique").to_numpy() > 1
    return RawRows(
        raw.X.iloc[groups["first"].to_numpy()].reset_index(drop=True),
        groups["family"].to_numpy(dtype=object),
        groups["flow"].to_numpy(),
        groups["size"].to_numpy(),
        conflicting,
    )


def evaluate_raw(
    model: RandomForestBaseline,
    rows: RawRows,
    target_fpr: float,
    attack_scores: np.ndarray,
    benign_scores: np.ndarray,
    attack: np.ndarray,
    benign: np.ndarray,
) -> dict[str, float]:
    """Score traffic with its repeats restored, using the threshold from cleaned validation rows.

    Every row of the held-out family is unseen. Benign rows are copies of cleaned test rows,
    plus rows that cleaning removed for matching another family, which no model trained on.
    """
    threshold = threshold_at_fpr(model.val_benign_scores, target_fpr)
    attack_count, benign_count = rows.count[attack], rows.count[benign]
    oracle = threshold_at_fpr(np.repeat(benign_scores, benign_count), target_fpr)
    conflict = rows.conflicting[attack]
    caught = attack_scores > threshold
    return {
        "raw_recall": float(attack_count[caught].sum() / attack_count.sum()),
        "raw_fpr": float(benign_count[benign_scores > threshold].sum() / benign_count.sum()),
        "raw_recall_at_oracle_fpr": float(attack_count[attack_scores > oracle].sum() / attack_count.sum()),
        "raw_distinct_recall": float(caught.mean()),
        "raw_attack_rows": int(attack_count.sum()),
        "raw_benign_rows": int(benign_count.sum()),
        "raw_conflicting_share": float(attack_count[conflict].sum() / attack_count.sum()),
    }


def without_suspect_benign(fold: Fold, suspect: np.ndarray) -> Fold:
    """Remove the suspect rows from every part of a fold. Attack rows are unaffected."""
    return Fold(
        fold.name, fold.train[~suspect[fold.train]], fold.val[~suspect[fold.val]],
        fold.test[~suspect[fold.test]], fold.held_out,
    )


def _rows_hash(*parts: np.ndarray) -> str:
    digest = hashlib.sha256()
    for part in parts:
        digest.update(np.asarray(part, dtype=np.int64).tobytes())
    return digest.hexdigest()[:16]


def _fold_rows(
    ds: Dataset,
    treatment: Treatment,
    fold: Fold,
    seed: int,
    config: BaselineConfig,
    flow_signatures: np.ndarray,
    indicators: pd.DataFrame,
    raw_rows: RawRows,
    flows: np.ndarray,
    suspect: np.ndarray | None = None,
) -> list[dict]:
    full_train_rows = len(fold.train)
    if treatment.exclude_suspect:
        fold = without_suspect_benign(fold, suspect)
    train = training_rows(fold, ds, treatment, flow_signatures)
    val_benign = fold.val[ds.family[fold.val].astype(str) == BENIGN]
    train_frame = treated_features(ds, treatment, train, indicators)
    categorical = [c for c in ds.categorical if c in train_frame.columns] if not treatment.missingness_only else []
    numeric = [c for c in train_frame.columns if c not in categorical]
    start = time.perf_counter()
    model = RandomForestBaseline(config.rf, seed).fit(
        train_frame, ds.y[train], treated_features(ds, treatment, val_benign, indicators), numeric, categorical
    )
    fit_seconds = time.perf_counter() - start

    scores = model.scores(treated_features(ds, treatment, fold.test, indicators))
    y = (ds.family[fold.test].astype(str) != BENIGN).astype(np.int8)
    attack_rows = fold.test[y == 1]
    attack_rows_train = train[ds.family[train].astype(str) != BENIGN]
    matched = np.isin(flow_signatures[attack_rows], flow_signatures[attack_rows_train])

    attack = raw_rows.family.astype(str) == fold.held_out
    test_benign = flows[fold.test[y == 0]]
    benign = (raw_rows.family.astype(str) == BENIGN) & (raw_rows.conflicting | np.isin(raw_rows.flow, test_benign))
    if treatment.exclude_suspect:
        benign &= ~raw_rows.suspect
    attack_scores = model.scores(treat_frame(raw_rows.X[attack], treatment, raw_rows.indicators[attack]))
    benign_scores = model.scores(treat_frame(raw_rows.X[benign], treatment, raw_rows.indicators[benign]))
    rows_tested = {
        "attack_rows_hash": _rows_hash(attack_rows, np.flatnonzero(attack)),
        "benign_rows_hash": _rows_hash(fold.test[y == 0], np.flatnonzero(benign)),
    }

    rows = []
    for target_fpr in config.target_fprs:
        threshold = threshold_at_fpr(model.val_benign_scores, target_fpr)
        caught = scores[y == 1] > threshold
        row = {
            "dataset": ds.name,
            "seed": seed,
            "treatment": treatment.name,
            "family": fold.held_out,
            "target_fpr": target_fpr,
            "train_rows": len(train),
            "train_rows_removed": full_train_rows - len(train),
            "n_attack": int(y.sum()),
            "n_benign": int((y == 0).sum()),
            "fit_seconds": fit_seconds,
            "auc": float(roc_auc_score(y, scores)),
            "signature_matched_rows": int(matched.sum()),
            "recall_signature_matched": float(caught[matched].mean()) if matched.any() else float("nan"),
            "recall_signature_unmatched": float(caught[~matched].mean()) if (~matched).any() else float("nan"),
            **evaluate_scores(y, scores, threshold, target_fpr, config.attack_prevalences),
            **evaluate_raw(model, raw_rows, target_fpr, attack_scores, benign_scores, attack, benign),
            **rows_tested,
        }
        rows.append(row)
    return rows


def meets_shortcut_rule(mean_drop: float, ci_low: float) -> bool:
    """Recall fell by at least the minimum, and the interval excludes zero on the side of a decrease."""
    return bool(mean_drop >= SHORTCUT_MIN_DROP and ci_low > 0)


def _treatment_role(treatment: str) -> str:
    if treatment in RULE_TREATMENTS:
        return "rule"
    return "label_sensitivity" if treatment == LABEL_SENSITIVITY else "diagnostic"


def _check_identical_rows(merged: pd.DataFrame, treatment: str) -> None:
    """Refuse a comparison made on different rows. Only the label sensitivity run changes benign rows."""
    columns = ["attack_rows_hash"] if treatment == LABEL_SENSITIVITY else ["attack_rows_hash", "benign_rows_hash"]
    for column in columns:
        if (merged[f"{column}_reference"] != merged[f"{column}_treated"]).any():
            raise ValueError(f"{treatment} was not evaluated on the reference rows ({column})")


def summarize_treatments(results: pd.DataFrame, samples: int, primary_fpr: float) -> pd.DataFrame:
    """Compare each treatment with the reference on the same seeds, families and held-out rows.

    ``mean_diff`` is the mean per-seed drop in raw-row recall (reference minus treated), so a
    positive value is recall lost, and ``ci_low`` and ``ci_high`` are its 95% interval. The
    ``dedup_`` columns repeat this on deduplicated rows, as a secondary sensitivity analysis.
    The rule cannot fire where reference recall is under the minimum drop, so
    ``rule_testable`` separates those families from ones that passed.
    """
    keys = ["dataset", "family", "target_fpr"]
    reference = results[results["treatment"] == REFERENCE]
    summary = []
    for treatment, part in results[results["treatment"] != REFERENCE].groupby("treatment"):
        merged = reference.merge(part, on=[*keys, "seed"], suffixes=("_reference", "_treated"))
        _check_identical_rows(merged, treatment)
        role = _treatment_role(treatment)
        for key, group in merged.groupby(keys, sort=True):
            row = {**dict(zip(keys, key)), "treatment": treatment, "role": role}
            for metric in ("raw_recall", "recall", "auc"):
                row[f"{metric}_reference"] = float(group[f"{metric}_reference"].mean())
                row[f"{metric}_treated"] = float(group[f"{metric}_treated"].mean())
            row.update(
                paired_comparison(
                    group["raw_recall_reference"].to_numpy(), group["raw_recall_treated"].to_numpy(), samples=samples
                )
            )
            secondary = paired_comparison(
                group["recall_reference"].to_numpy(), group["recall_treated"].to_numpy(), samples=samples
            )
            row.update({f"dedup_{name}": secondary[name] for name in ("mean_diff", "ci_low", "ci_high")})
            applies = role == "rule" and bool(np.isclose(key[2], primary_fpr))
            row["rule_testable"] = row["raw_recall_reference"] >= SHORTCUT_MIN_DROP
            row["shortcut_dependent"] = applies and meets_shortcut_rule(row["mean_diff"], row["ci_low"])
            row["dedup_shortcut_dependent"] = applies and meets_shortcut_rule(
                row["dedup_mean_diff"], row["dedup_ci_low"]
            )
            summary.append(row)
    return pd.DataFrame(summary)


def rule_report(effects: pd.DataFrame, primary_fpr: float) -> dict:
    """State the rule and disclose how many comparisons it was applied to."""
    primary = effects[np.isclose(effects["target_fpr"], primary_fpr)]
    rule = primary[primary["role"] == "rule"]
    return {
        "evaluation_role": EVALUATION_ROLE,
        "primary_metric": "raw_recall",
        "secondary_metric": "recall on deduplicated rows",
        "primary_target_fpr": primary_fpr,
        "min_recall_drop": SHORTCUT_MIN_DROP,
        "interval": "95% percentile bootstrap of the mean paired per-seed drop; lower bound above zero",
        "multiple_comparison_correction": "none",
        "n_comparisons": int(len(rule)),
        "n_comparisons_by_dataset": {str(k): int(v) for k, v in rule["dataset"].value_counts().sort_index().items()},
        "n_comparisons_by_treatment": {str(k): int(v) for k, v in rule["treatment"].value_counts().sort_index().items()},
        "n_not_testable": int((~rule["rule_testable"]).sum()),
        "n_shortcut_dependent": int(rule["shortcut_dependent"].sum()),
        "n_shortcut_dependent_dedup": int(rule["dedup_shortcut_dependent"].sum()),
        "treatments_outside_the_rule": sorted(set(primary["treatment"]) - set(rule["treatment"])),
    }


def run(
    config: BaselineConfig,
    out: str | Path,
    extra_manifest: dict | None = None,
    log: Callable[[str], None] = print,
) -> pd.DataFrame:
    """Run or continue the artifact experiments, then write the combined result files."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    prepared = []
    suspects: dict[str, np.ndarray | None] = {}
    suspect_counts = {}
    for dataset_config in config.datasets:
        raw = load_dataset(dataset_config)
        raw_rows = build_raw_rows(raw)
        clean, stats = remove_duplicates(raw)
        del raw
        gc.collect()
        raw_rows.indicators = missingness_frame(
            Dataset(clean.name, raw_rows.X, raw_rows.family, clean.categorical)
        ).reindex(columns=missingness_frame(clean).columns, fill_value=0.0)
        suspect = suspect_benign_mask(clean)
        suspects[clean.name] = suspect
        if suspect is not None:
            flows = pd.util.hash_pandas_object(clean.X, index=False).to_numpy()
            raw_rows.suspect = (raw_rows.family.astype(str) == BENIGN) & np.isin(raw_rows.flow, flows[suspect])
            suspect_counts[clean.name] = {
                "cleaned_rows": int(suspect.sum()), "raw_rows": int(raw_rows.count[raw_rows.suspect].sum()),
            }
            del flows
        prepared.append((clean, stats, raw_rows))

    datasets = [p[0] for p in prepared]
    plans = {ds.name: treatments_for(ds, suspects[ds.name] is not None) for ds in datasets}
    extra = {
        "experiment": "artifact_resistance",
        "evaluation_role": EVALUATION_ROLE,
        "suspect_benign_excluded_in_sensitivity_only": suspect_counts,
        "treatments": {name: [t.name for t in plan] for name, plan in plans.items()},
        **(extra_manifest or {}),
    }
    _write_manifest(out, config, datasets, {p[0].name: p[1] for p in prepared}, extra, log)
    store = FoldStore(out / "folds")
    total = sum(
        len(plans[ds.name]) * len(eligible_families(ds.family, config.min_family_count)) for ds in datasets
    ) * len(config.seeds)
    progress = Progress(total, log)

    for seed in config.seeds:
        for ds, _, raw_rows in prepared:
            flow_signatures = signatures(ds)
            flows = pd.util.hash_pandas_object(ds.X, index=False).to_numpy()
            indicators = missingness_frame(ds)
            partition = make_partition(ds.family, seed, config.val_fraction, config.test_fraction)
            for family in eligible_families(ds.family, config.min_family_count):
                fold = holdout_fold(partition, ds.family, family, config.max_train_per_family, seed)
                for treatment in plans[ds.name]:
                    label = f"{treatment.name}:{family}"
                    if store.done(ds.name, seed, label):
                        progress.skip()
                        continue
                    started = time.perf_counter()
                    rows = _fold_rows(ds, treatment, fold, seed, config, flow_signatures, indicators, raw_rows, flows, suspects[ds.name])
                    store.save(ds.name, seed, label, rows)
                    progress.update(f"seed {seed} {ds.name} {label}", time.perf_counter() - started)
                    gc.collect()

    results = store.load_all()
    results = results[results["seed"].isin(config.seeds)].sort_values(ROW_KEYS).reset_index(drop=True)
    results.to_csv(out / "results.csv", index=False)
    primary_fpr = config.target_fprs[0]
    effects = summarize_treatments(results, config.bootstrap_samples, primary_fpr)
    effects.to_csv(out / "treatment_effects.csv", index=False)
    (out / "shortcut_rule.json").write_text(json.dumps(rule_report(effects, primary_fpr), indent=2), encoding="utf-8")
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True)
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    run(load_config(args.config), args.out)
    print(f"Results written to {args.out}")


if __name__ == "__main__":
    main()
''',
    'aegis/experiments/baseline.py': r'''"""Run Config A: seen-family and held-out-family evaluation, plus cross-dataset tests if enabled.

    python -m aegis.experiments.baseline --config configs/baseline.yaml --out results/config_a

Each finished fold is saved on its own, so running again with the same output folder
continues where an interrupted run stopped. Seeds can also be added this way.
"""

from __future__ import annotations

import argparse
import dataclasses
import datetime
import gc
import json
import os
import platform
import re
import time
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import sklearn
import torch

from aegis.config import BaselineConfig, load_config
from aegis.data.dataset import BENIGN, Dataset, remove_duplicates
from aegis.data.loaders import dataset_files, load_dataset, project_to_shared
from aegis.data.splits import (
    Fold,
    cap_per_family,
    eligible_families,
    holdout_fold,
    make_partition,
    seen_fold,
)
from aegis.evaluation.metrics import (
    bootstrap_ci,
    evaluate_scores,
    paired_comparison,
    threshold_at_fpr,
)
from aegis.models.baseline import SCORERS, RFAEBaseline
from aegis.provenance import file_sha256, git_state, source_hash

ALL_ATTACKS = "all_attacks"
ID_COLUMNS = ["dataset", "test_dataset", "seed", "fold", "eval_family", "scorer", "target_fpr"]
HOLDOUT_PREFIX = "holdout:"


class FoldStore:
    """One CSV per finished fold. Files appear atomically, so none is ever half written."""

    def __init__(self, directory: str | Path) -> None:
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)

    def _path(self, dataset: str, seed: int, fold: str) -> Path:
        name = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{dataset}__seed{seed}__{fold}")
        return self.directory / f"{name}.csv"

    def done(self, dataset: str, seed: int, fold: str) -> bool:
        return self._path(dataset, seed, fold).exists()

    def save(self, dataset: str, seed: int, fold: str, rows: list[dict]) -> None:
        path = self._path(dataset, seed, fold)
        temporary = path.with_suffix(".tmp")
        pd.DataFrame(rows).to_csv(temporary, index=False)
        os.replace(temporary, path)

    def load_all(self) -> pd.DataFrame:
        return pd.concat(
            [pd.read_csv(p) for p in sorted(self.directory.glob("*.csv"))], ignore_index=True
        )


class Progress:
    """Print one line per fold, with an estimate of the time left."""

    def __init__(self, total: int, log: Callable[[str], None]) -> None:
        self.total, self.log = total, log
        self.finished, self.fitted, self.seconds = 0, 0, 0.0

    def skip(self) -> None:
        self.finished += 1

    def update(self, label: str, seconds: float) -> None:
        self.finished += 1
        self.fitted += 1
        self.seconds += seconds
        left = (self.seconds / self.fitted) * (self.total - self.finished) / 3600
        self.log(
            f"[{self.finished}/{self.total}] {label}: {seconds / 60:.1f} min; "
            f"about {left:.1f} h left at this session's average"
        )


def _score_rows(
    model: RFAEBaseline,
    fit_seconds: float,
    source: Dataset,
    target: Dataset,
    test_rows: np.ndarray,
    fold_name: str,
    eval_families: list[str],
    config: BaselineConfig,
    seed: int,
) -> list[dict]:
    """Evaluate one fitted model. Thresholds come only from validation benign scores."""
    scores = model.scores(target.X.iloc[test_rows])
    families = target.family[test_rows].astype(str)
    rows = []
    for eval_family in eval_families:
        if eval_family == ALL_ATTACKS:
            mask = np.ones(len(families), dtype=bool)
        else:
            mask = np.isin(families, [BENIGN, eval_family])
        y = (families[mask] != BENIGN).astype(np.int8)
        for scorer in SCORERS:
            for target_fpr in config.target_fprs:
                threshold = threshold_at_fpr(model.val_benign_scores[scorer], target_fpr)
                rows.append(
                    {
                        "dataset": source.name,
                        "test_dataset": target.name,
                        "seed": seed,
                        "fold": fold_name,
                        "eval_family": eval_family,
                        "scorer": scorer,
                        "target_fpr": target_fpr,
                        "n_attack": int(y.sum()),
                        "n_benign": int((y == 0).sum()),
                        "fit_seconds": fit_seconds,
                        **evaluate_scores(
                            y, scores[scorer][mask], threshold, target_fpr, config.attack_prevalences
                        ),
                    }
                )
    return rows


def _run_fold(
    source: Dataset,
    target: Dataset,
    fold: Fold,
    test_rows: np.ndarray,
    eval_families: list[str],
    config: BaselineConfig,
    seed: int,
    store: FoldStore,
    progress: Progress,
) -> None:
    if store.done(source.name, seed, fold.name):
        progress.skip()
        return
    start = time.perf_counter()
    model = RFAEBaseline(config.rf, config.autoencoder, seed).fit(
        source.X.iloc[fold.train],
        source.y[fold.train],
        source.X.iloc[fold.val],
        source.y[fold.val],
        source.numeric,
        source.categorical,
    )
    fit_seconds = time.perf_counter() - start
    rows = _score_rows(
        model, fit_seconds, source, target, test_rows, fold.name, eval_families, config, seed
    )
    store.save(source.name, seed, fold.name, rows)
    progress.update(f"seed {seed} {source.name} {fold.name}", time.perf_counter() - start)
    del model
    gc.collect()


def evaluate_within_dataset(
    ds: Dataset, config: BaselineConfig, seed: int, store: FoldStore, progress: Progress
) -> None:
    partition = make_partition(ds.family, seed, config.val_fraction, config.test_fraction)
    families = eligible_families(ds.family, config.min_family_count)
    if not families:
        raise ValueError(f"{ds.name}: no family reaches min_family_count={config.min_family_count}")

    fold = seen_fold(partition, ds.family, config.max_train_per_family, seed)
    _run_fold(ds, ds, fold, fold.test, [ALL_ATTACKS, *families], config, seed, store, progress)
    for family in families:
        fold = holdout_fold(partition, ds.family, family, config.max_train_per_family, seed)
        _run_fold(ds, ds, fold, fold.test, [family], config, seed, store, progress)


def evaluate_cross_dataset(
    source: Dataset,
    target: Dataset,
    config: BaselineConfig,
    seed: int,
    store: FoldStore,
    progress: Progress,
) -> None:
    """Train and calibrate on one dataset, then test on every row of the other."""
    partition = make_partition(source.family, seed, config.val_fraction, config.test_fraction)
    train = cap_per_family(partition.train, source.family, config.max_train_per_family, seed)
    fold = Fold(f"cross:{source.name}->{target.name}", train, partition.val, partition.test)
    rows = np.arange(len(target.X))
    _run_fold(source, target, fold, rows, [ALL_ATTACKS], config, seed, store, progress)


def summarize(results: pd.DataFrame, samples: int) -> pd.DataFrame:
    """Mean and bootstrap interval across seeds for each metric."""
    group = ["dataset", "test_dataset", "fold", "eval_family", "scorer", "target_fpr"]
    metrics = [c for c in results.columns if c not in ID_COLUMNS and c not in ("n_attack", "n_benign")]
    rows = []
    for key, part in results.groupby(group, sort=True):
        row = dict(zip(group, key), n_seeds=part["seed"].nunique())
        for metric in metrics:
            low, high = bootstrap_ci(part[metric].to_numpy(), samples=samples)
            row[f"{metric}_mean"] = float(part[metric].mean())
            row[f"{metric}_ci_low"], row[f"{metric}_ci_high"] = low, high
        rows.append(row)
    return pd.DataFrame(rows)


def degradation(results: pd.DataFrame, samples: int) -> pd.DataFrame:
    """RQ1 reference: recall lost on the same test flows once a family is held out."""
    keys = ["dataset", "eval_family", "scorer", "target_fpr"]
    within = results[results["dataset"] == results["test_dataset"]]
    seen = within[(within["fold"] == "seen") & (within["eval_family"] != ALL_ATTACKS)]
    held = within[within["fold"].str.startswith(HOLDOUT_PREFIX)]
    merged = seen.merge(held, on=[*keys, "seed"], suffixes=("_seen", "_heldout"))
    rows = []
    for key, part in merged.groupby(keys, sort=True):
        rows.append(
            {
                **dict(zip(keys, key)),
                "recall_seen_mean": float(part["recall_seen"].mean()),
                "recall_heldout_mean": float(part["recall_heldout"].mean()),
                **paired_comparison(
                    part["recall_seen"].to_numpy(), part["recall_heldout"].to_numpy(), samples=samples
                ),
            }
        )
    return pd.DataFrame(rows)


def phase1_gate(results: pd.DataFrame, datasets: list[Dataset], config: BaselineConfig) -> pd.DataFrame:
    """Roadmap gate for phase 1: Config A measured on every eligible held-out family."""
    rows = []
    for ds in datasets:
        for family in eligible_families(ds.family, config.min_family_count):
            part = results[
                (results["dataset"] == ds.name)
                & (results["test_dataset"] == ds.name)
                & (results["fold"] == f"{HOLDOUT_PREFIX}{family}")
                & (results["eval_family"] == family)
            ]
            measured = sorted(int(s) for s in part["seed"].unique() if s in config.seeds)
            rows.append(
                {
                    "dataset": ds.name,
                    "family": family,
                    "seeds_measured": len(measured),
                    "seeds_required": len(config.seeds),
                    "complete": len(measured) == len(config.seeds),
                }
            )
    return pd.DataFrame(rows)


def prepare_datasets(config: BaselineConfig) -> tuple[list[Dataset], dict[str, dict[str, int]]]:
    """Load and clean every configured dataset, returning the cleaning counts too."""
    datasets, cleaning = [], {}
    for dataset_config in config.datasets:
        cleaned, stats = remove_duplicates(load_dataset(dataset_config))
        datasets.append(cleaned)
        cleaning[cleaned.name] = stats
    return datasets, cleaning


def _experiment_settings(config: BaselineConfig) -> dict:
    """Settings that define the experiment, leaving out what describes the machine.

    Seeds and the cross-dataset switch only add folds. Worker count, device and data
    location can change when a session moves to another runtime.
    """
    settings = dataclasses.asdict(config)
    for key in ("seeds", "cross_dataset"):
        settings.pop(key)
    settings["rf"].pop("n_jobs")
    settings["autoencoder"].pop("device")
    for dataset in settings["datasets"]:
        dataset.pop("path")
    return settings


def _source_checksums(config: BaselineConfig) -> dict[str, dict[str, str]]:
    """SHA-256 of each source CSV, read once per session whether or not folds remain."""
    checksums = {}
    for dataset in config.datasets:
        files = dataset_files(dataset)
        if files:
            checksums[dataset.name or dataset.kind] = {p.name: file_sha256(p) for p in files}
    return checksums


def _write_manifest(
    out: Path,
    config: BaselineConfig,
    datasets: list[Dataset],
    cleaning: dict,
    extra: dict | None,
    log: Callable[[str], None],
) -> None:
    """Record what produced the results, and refuse to mix incomparable runs.

    The identity covers the settings, package source and data contents, since a change to
    any of them makes saved folds incomparable with new ones. Source checksums are part of
    it because counts cannot see a release that changes values but not row counts. Files
    are identified by name and content, not location, so a moved folder still resumes.
    Each session's machine details are logged, because GPU and CPU autoencoder training
    can differ numerically.
    """
    family_counts = {d.name: d.family_counts() for d in datasets}
    identity = json.loads(
        json.dumps(
            {
                "settings": _experiment_settings(config),
                "source_version": source_hash(),
                "data": {
                    "cleaning": cleaning,
                    "family_counts": family_counts,
                    "source_sha256": _source_checksums(config),
                },
                "extra": extra or {},
            }
        )
    )

    sessions: list[dict] = []
    path = out / "manifest.json"
    if path.exists():
        try:
            previous = json.loads(path.read_text(encoding="utf-8"))
            previous_identity, sessions = previous["identity"], previous["sessions"]
        except (ValueError, KeyError, TypeError) as error:
            raise ValueError(
                f"{path} is unreadable or from an older version of Aegis, so its results "
                "cannot be checked for comparability. Use a new output folder."
            ) from error
        if previous_identity != identity:
            raise ValueError(
                f"{out} holds results from different settings, code or data. "
                "Use a new output folder."
            )
    device = config.autoencoder.device
    if sessions and sessions[-1]["ae_device"] != device:
        log(
            f"NOTE: autoencoder device changed from {sessions[-1]['ae_device']} to {device}. "
            "Saved folds are kept, and manifest.json records each session's device."
        )
    sessions.append(
        {
            "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "folds_already_done": len(list((out / "folds").glob("*.csv"))),
            "ae_device": device,
            "rf_n_jobs": config.rf.n_jobs,
            "dataset_paths": {d.name or d.kind: d.path for d in config.datasets},
            "git": git_state(),
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
            "torch": torch.__version__,
        }
    )
    path.write_text(
        json.dumps(
            {
                "identity": identity,
                "seeds": list(config.seeds),
                "cross_dataset": config.cross_dataset,
                "sessions": sessions,
                "cleaning": cleaning,
                "family_counts": family_counts,
                "held_out_families": {
                    d.name: eligible_families(d.family, config.min_family_count) for d in datasets
                },
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def run(
    config: BaselineConfig,
    out: str | Path,
    prepared: tuple[list[Dataset], dict] | None = None,
    extra_manifest: dict | None = None,
    log: Callable[[str], None] = print,
) -> pd.DataFrame:
    """Run or continue an experiment, then write the combined result files."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    datasets, cleaning = prepared if prepared is not None else prepare_datasets(config)
    _write_manifest(out, config, datasets, cleaning, extra_manifest, log)
    store = FoldStore(out / "folds")

    cross = config.cross_dataset and len(datasets) >= 2
    within_folds = sum(
        1 + len(eligible_families(d.family, config.min_family_count)) for d in datasets
    )
    cross_folds = len(datasets) * (len(datasets) - 1) if cross else 0
    progress = Progress((within_folds + cross_folds) * len(config.seeds), log)

    for seed in config.seeds:
        for ds in datasets:
            evaluate_within_dataset(ds, config, seed, store, progress)

    if cross:
        shared = [remove_duplicates(project_to_shared(d))[0] for d in datasets]
        for seed in config.seeds:
            for source in shared:
                for target in shared:
                    if source.name != target.name:
                        evaluate_cross_dataset(source, target, config, seed, store, progress)

    results = store.load_all()
    results = results[results["seed"].isin(config.seeds)].reset_index(drop=True)
    results.to_csv(out / "results.csv", index=False)
    summarize(results, config.bootstrap_samples).to_csv(out / "summary.csv", index=False)
    degradation(results, config.bootstrap_samples).to_csv(out / "degradation.csv", index=False)
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True)
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    run(load_config(args.config), args.out)
    print(f"Results written to {args.out}")


if __name__ == "__main__":
    main()
''',
    'aegis/models/__init__.py': r'''''',
    'aegis/models/autoencoder.py': r'''"""Autoencoder trained on benign flows; high reconstruction error marks a flow as unusual."""

from __future__ import annotations

import numpy as np
import torch
from torch import nn

from aegis.config import AutoencoderConfig


def _layers(sizes: list[int]) -> list[nn.Module]:
    layers: list[nn.Module] = []
    for size_in, size_out in zip(sizes, sizes[1:]):
        layers += [nn.Linear(size_in, size_out), nn.ReLU()]
    return layers[:-1]  # The final layer stays linear.


class AutoencoderScorer:
    def __init__(self, config: AutoencoderConfig, seed: int) -> None:
        self.config = config
        self.seed = seed
        self.model: nn.Module | None = None

    def fit(self, benign_train: np.ndarray, benign_val: np.ndarray) -> "AutoencoderScorer":
        cfg = self.config
        torch.manual_seed(self.seed)
        generator = np.random.default_rng(self.seed)
        sizes = [benign_train.shape[1], *cfg.hidden, cfg.latent]
        self.model = nn.Sequential(*_layers(sizes), nn.ReLU(), *_layers(sizes[::-1])).to(cfg.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=cfg.learning_rate)
        data = torch.as_tensor(benign_train, dtype=torch.float32)

        best_loss, best_state, stale = float("inf"), None, 0
        for _ in range(cfg.epochs):
            self.model.train()
            order = generator.permutation(len(data))
            for start in range(0, len(data), cfg.batch_size):
                batch = data[order[start : start + cfg.batch_size]].to(cfg.device)
                optimizer.zero_grad()
                nn.functional.mse_loss(self.model(batch), batch).backward()
                optimizer.step()

            val_loss = float(self.reconstruction_error(benign_val).mean())
            if val_loss < best_loss - 1e-6:
                best_loss, stale = val_loss, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                stale += 1
                if stale >= cfg.patience:
                    break
        if best_state is None:
            raise RuntimeError("Autoencoder training did not produce a finite validation loss")
        self.model.load_state_dict(best_state)
        return self

    @torch.no_grad()
    def reconstruction_error(self, X: np.ndarray) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Call fit before scoring")
        self.model.eval()
        errors = []
        data = torch.as_tensor(X, dtype=torch.float32)
        for start in range(0, len(data), 8192):
            batch = data[start : start + 8192].to(self.config.device)
            errors.append(((self.model(batch) - batch) ** 2).mean(dim=1).cpu().numpy())
        return np.concatenate(errors)
''',
    'aegis/models/baseline.py': r'''"""Config A defender: a static Random Forest plus autoencoder.

Inspired by Agrawal et al. (2026), not a replication: the paper does not release its
preprocessing or model settings. The fusion rule below is an Aegis design choice.
"""

from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from aegis.config import AutoencoderConfig, RandomForestConfig
from aegis.models.autoencoder import AutoencoderScorer

SCORERS = ("rf", "ae", "fused")


def signed_log1p(values: np.ndarray) -> np.ndarray:
    """Compress heavy-tailed flow counts while keeping the sign of negative values."""
    return np.sign(values) * np.log1p(np.abs(values))


def build_preprocessor(
    numeric: Sequence[str], categorical: Sequence[str]
) -> ColumnTransformer:
    numeric_steps = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("log", FunctionTransformer(signed_log1p)),
            ("scale", StandardScaler()),
        ]
    )
    categorical_steps = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=20,
        sparse_output=False,
        dtype=np.float32,
    )
    transformers = [("numeric", numeric_steps, list(numeric))]
    if categorical:
        transformers.append(("categorical", categorical_steps, list(categorical)))
    return ColumnTransformer(transformers, sparse_threshold=0.0)


class TailCalibrator:
    """Express a score as how rare it is among validation benign traffic.

    The result is -log10 of the share of benign scores at or above the value, so two
    scorers with very different distributions become comparable. Standardizing is not
    enough: one scorer's long benign tail can set the fused threshold above everything
    the other scorer finds. Scores beyond the benign maximum keep their order through a
    term that grows with the distance past that maximum.
    """

    def __init__(self, benign_scores: np.ndarray) -> None:
        self.sorted = np.sort(benign_scores)
        self.spread = max(float(benign_scores.std()), 1e-6)

    def __call__(self, scores: np.ndarray) -> np.ndarray:
        n = len(self.sorted)
        at_or_above = n - np.searchsorted(self.sorted, scores, side="left")
        rarity = -np.log10((at_or_above + 1) / (n + 1))
        beyond = np.maximum(scores - self.sorted[-1], 0.0) / self.spread
        return rarity + beyond


def fit_forest(config: RandomForestConfig, seed: int, X: np.ndarray, y: np.ndarray) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        min_samples_leaf=config.min_samples_leaf,
        n_jobs=config.n_jobs,
        class_weight="balanced_subsample",  # Used instead of SMOTE.
        random_state=seed,
    ).fit(X, y)


def forest_log_odds(forest: RandomForestClassifier, X: np.ndarray) -> np.ndarray:
    probability = forest.predict_proba(X)[:, list(forest.classes_).index(1)]
    clipped = np.clip(probability, 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped))


class RandomForestBaseline:
    """The Config A baseline score: the forest alone, with no autoencoder to fit."""

    def __init__(self, rf: RandomForestConfig, seed: int) -> None:
        self.rf_config = rf
        self.seed = seed
        self.val_benign_scores = np.empty(0)

    def fit(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_val_benign: pd.DataFrame,
        numeric: Sequence[str],
        categorical: Sequence[str] = (),
    ) -> "RandomForestBaseline":
        if len(np.unique(y_train)) != 2:
            raise ValueError("Training data must contain both benign and attack rows")
        self.preprocessor = build_preprocessor(numeric, categorical).fit(X_train)
        train = self.preprocessor.transform(X_train).astype(np.float32)
        self.forest = fit_forest(self.rf_config, self.seed, train, y_train)
        self.val_benign_scores = self.scores(X_val_benign)
        return self

    def scores(self, X: pd.DataFrame) -> np.ndarray:
        return forest_log_odds(self.forest, self.preprocessor.transform(X).astype(np.float32))


class RFAEBaseline:
    """Combine a supervised score with an anomaly score.

    The RF score is a log-odds value and the AE score a log error. The fused score is
    the larger of their calibrated values, so a flow can be flagged for looking like a
    known attack or for looking unusual.
    """

    def __init__(
        self, rf: RandomForestConfig, autoencoder: AutoencoderConfig, seed: int
    ) -> None:
        self.rf_config = rf
        self.ae_config = autoencoder
        self.seed = seed
        self.val_benign_scores: dict[str, np.ndarray] = {}

    def fit(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_val: pd.DataFrame,
        y_val: np.ndarray,
        numeric: Sequence[str],
        categorical: Sequence[str] = (),
    ) -> "RFAEBaseline":
        if len(np.unique(y_train)) != 2:
            raise ValueError("Training data must contain both benign and attack rows")

        # Every fitted statistic comes from the training rows.
        self.preprocessor = build_preprocessor(numeric, categorical).fit(X_train)
        train = self.preprocessor.transform(X_train).astype(np.float32)
        val = self.preprocessor.transform(X_val).astype(np.float32)

        self.forest = fit_forest(self.rf_config, self.seed, train, y_train)

        val_benign = val[y_val == 0]
        self.autoencoder = AutoencoderScorer(self.ae_config, self.seed).fit(
            train[y_train == 0], val_benign
        )

        rf_raw, ae_raw = self._raw_scores(val_benign)
        self._rf_calibrator = TailCalibrator(rf_raw)
        self._ae_calibrator = TailCalibrator(ae_raw)
        self.val_benign_scores = self._combine(rf_raw, ae_raw)
        return self

    def _raw_scores(self, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        rf = forest_log_odds(self.forest, X)
        ae = np.log(self.autoencoder.reconstruction_error(X) + 1e-12)
        return rf, ae

    def _combine(self, rf: np.ndarray, ae: np.ndarray) -> dict[str, np.ndarray]:
        fused = np.maximum(self._rf_calibrator(rf), self._ae_calibrator(ae))
        return {"rf": rf, "ae": ae, "fused": fused}

    def scores(self, X: pd.DataFrame) -> dict[str, np.ndarray]:
        transformed = self.preprocessor.transform(X).astype(np.float32)
        return self._combine(*self._raw_scores(transformed))
''',
    'aegis/provenance.py': r'''"""Record which code produced a run. Standard library only, so the notebook builder can use it."""

from __future__ import annotations

import hashlib
import subprocess
from pathlib import Path

PACKAGE_DIR = Path(__file__).resolve().parent


def source_files(root: str | Path = PACKAGE_DIR.parent) -> dict[str, str]:
    """Return the package's Python files by path relative to ``root``, the package's parent."""
    root = Path(root)
    return {
        path.relative_to(root).as_posix(): path.read_text(encoding="utf-8")
        for path in sorted((root / PACKAGE_DIR.name).rglob("*.py"))
    }


def source_hash(root: str | Path = PACKAGE_DIR.parent) -> str:
    """Fingerprint the package source. Works with or without git, including in the notebook."""
    digest = hashlib.sha256()
    for name, text in source_files(root).items():
        digest.update(name.encode())
        digest.update(text.encode())
    return digest.hexdigest()[:16]


def file_sha256(path: str | Path, chunk_size: int = 1 << 20) -> str:
    """Hash a file in chunks, so a large CSV is never held in memory."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def git_state(directory: str | Path = PACKAGE_DIR) -> str:
    """Describe the commit holding ``directory``, or ``unknown`` outside a repository.

    ``-dirty`` means files under ``directory`` differ from that commit. ``git describe
    --dirty`` alone ignores untracked files, so it would call uncommitted code clean.
    """

    def git(*args: str) -> str:
        return subprocess.run(
            ["git", "-C", str(directory), *args], capture_output=True, text=True, check=True
        ).stdout.strip()

    try:
        commit = git("describe", "--always")
        return f"{commit}-dirty" if git("status", "--porcelain", "--", ".") else commit
    except (OSError, subprocess.CalledProcessError):
        return "unknown"
''',
}

source_root = Path(tempfile.mkdtemp(prefix='aegis_source_'))
for name, text in FILES.items():
    path = source_root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
sys.path.insert(0, str(source_root))
print('Aegis source version', SOURCE_VERSION)


In [ ]:
import multiprocessing, platform
import numpy, pandas, scipy, sklearn, torch

try:
    ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
except (AttributeError, ValueError, OSError):
    ram_gb = float('nan')
cpus = multiprocessing.cpu_count()
AE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'platform     {platform.platform()}')
print(f'python       {platform.python_version()}')
print(f'numpy        {numpy.__version__}')
print(f'pandas       {pandas.__version__}')
print(f'scipy        {scipy.__version__}')
print(f'scikit-learn {sklearn.__version__}')
print(f'torch        {torch.__version__}')
print(f'cpus         {cpus}')
print(f'ram_gb       {ram_gb:.1f}')
print(f'ae_device    {AE_DEVICE}')
if ram_gb < 12:
    print('WARNING: under 12 GB of RAM. If the run crashes, lower MAX_TRAIN_PER_FAMILY '
          'and use a new OUTPUT_DIR.')


## 3. Check the pipeline on generated data

In [ ]:
import tempfile
from aegis.config import AutoencoderConfig, BaselineConfig, DatasetConfig, RandomForestConfig
from aegis.experiments.artifacts import run as run_artifacts

check_config = BaselineConfig(
    datasets=(DatasetConfig(kind='synthetic', name='synthetic_a', seed=11),),
    seeds=(0,), max_train_per_family=5000, min_family_count=100,
    target_fprs=TARGET_FPRS, attack_prevalences=ATTACK_PREVALENCES, bootstrap_samples=200,
    rf=RandomForestConfig(n_estimators=20, max_depth=10, n_jobs=cpus),
)
with tempfile.TemporaryDirectory() as folder:
    assert len(run_artifacts(check_config, folder, log=lambda message: None)), 'The pipeline check produced no rows'
print('Pipeline check passed on generated data. These numbers say nothing about real traffic.')


## 4. Find the real data

In [ ]:
from aegis.data.loaders import locate_data

located = locate_data(DATA_ROOT) if os.path.isdir(DATA_ROOT) else {}
for name, folder in located.items():
    print(f'{name}: {folder}')
if len(located) < 2:
    absent = sorted({'cicids2017', 'unsw_nb15'} - set(located))
    raise FileNotFoundError(
        f'Could not find {absent} under {DATA_ROOT}. See the first cell for the expected files.'
    )

config = BaselineConfig(
    datasets=(
        # A port identifies the service, so it can stand in for the attack family.
        DatasetConfig(kind='cicids2017', path=str(located['cicids2017']),
                      drop_columns=('Destination Port',)),
        DatasetConfig(kind='unsw_nb15', path=str(located['unsw_nb15'])),
    ),
    seeds=tuple(SEEDS),
    cross_dataset=RUN_CROSS_DATASET,
    max_train_per_family=MAX_TRAIN_PER_FAMILY,
    min_family_count=MIN_FAMILY_COUNT,
    target_fprs=TARGET_FPRS,
    attack_prevalences=ATTACK_PREVALENCES,
    rf=RandomForestConfig(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=cpus),
    autoencoder=AutoencoderConfig(device=AE_DEVICE),
)


## 5. Run the experiments
This is the long step. A progress line appears after each fold.

In [ ]:
import time

started = time.time()
results = run_artifacts(config, OUTPUT_DIR, extra_manifest={'source_version': SOURCE_VERSION})
print(f'Finished in {(time.time() - started) / 3600:.2f} h this session; {len(results)} result rows')


## 6. Compare with Config A

In [ ]:
import json
import pandas as pd

previous_manifest = os.path.join(PREVIOUS_DIR, 'manifest.json')
previous_results = os.path.join(PREVIOUS_DIR, 'results.csv')
if os.path.exists(previous_manifest) and os.path.exists(previous_results):
    with open(previous_manifest, encoding='utf-8') as handle:
        earlier = json.load(handle)
    with open(os.path.join(OUTPUT_DIR, 'manifest.json'), encoding='utf-8') as handle:
        current = json.load(handle)
    same = earlier['identity']['data']['source_sha256'] == current['identity']['data']['source_sha256']
    print('Same source data as Config A:', same)
    old = pd.read_csv(previous_results)
    old = old[(old['scorer'] == 'rf') & old['fold'].str.startswith('holdout:')]
    new = results[results['treatment'] == 'reference']
    paired = new.merge(old, left_on=['dataset', 'family', 'seed', 'target_fpr'],
                       right_on=['dataset', 'eval_family', 'seed', 'target_fpr'], suffixes=('', '_config_a'))
    gap = (paired['recall'] - paired['recall_config_a']).abs()
    print(f'Reference recall vs Config A RF: mean gap {gap.mean():.4f}, largest {gap.max():.4f}, {len(paired)} pairs.')
    print('A gap means the library versions or machine differ. Compare treatments within this run only.')
else:
    print('No Config A results found in', PREVIOUS_DIR, 'so this comparison was skipped.')


## 7. Results

In [ ]:
pd.set_option('display.max_rows', 300, 'display.width', 200, 'display.max_columns', 40)
primary_fpr = config.target_fprs[0]
main = results[results['target_fpr'] == primary_fpr]
reference = main[main['treatment'] == 'reference']
effects = pd.read_csv(f'{OUTPUT_DIR}/treatment_effects.csv')
effects = effects[effects['target_fpr'] == primary_fpr]
with open(f'{OUTPUT_DIR}/shortcut_rule.json', encoding='utf-8') as handle:
    rule = json.load(handle)
EFFECT_COLUMNS = ['dataset', 'family', 'treatment', 'raw_recall_reference', 'raw_recall_treated', 'mean_diff',
                  'ci_low', 'ci_high', 'rule_testable', 'shortcut_dependent']


def show_effects(title, treatments, family=None):
    # mean_diff is reference minus treated raw recall, so a negative value is recall gained.
    print(title)
    table = effects[effects['treatment'].isin(treatments)]
    if family is not None:
        table = table[table['family'] == family]
    display(table[EFFECT_COLUMNS].round(3))


print('This is a sensitivity audit, not an unbiased generalization estimate.')
show_effects('1. Web attack recall after signature removal', ['remove_matching_signatures'], 'web_attack')

print('2. PortScan on raw published rows, reference treatment')
display(reference[reference['family'] == 'portscan'].groupby('dataset')[
    ['raw_recall', 'recall', 'raw_recall_at_oracle_fpr', 'raw_fpr', 'raw_distinct_recall',
     'raw_conflicting_share', 'raw_attack_rows']].mean().round(3))

show_effects('3. Sensitivity to TCP window features', ['no_tcp_window'])
show_effects('4. Signal through markers (missingness_only sees nothing else)',
             ['neutralize_missing_markers', 'missingness_only'])
print('4b. UNSW-NB15 service: kept with its catch-all value, removed, or reduced to one flag')
service = main[(main['dataset'] == 'unsw_nb15')
               & main['treatment'].isin(['reference', 'neutralize_missing_markers', 'service_marker_only'])]
display(service.pivot_table(index='family', columns='treatment', values='raw_recall').round(3))
show_effects('5. Removing ct_* context counts', ['no_context_counts'])
show_effects('6. all_controls across unseen attack families', ['all_controls'])

print('7. Possibly mislabeled benign rows: original labels are primary, and this run excludes them')
print('Attack rows are identical to the reference. Benign rows differ by design, so raw_fpr is shown too.')
suspect = effects[effects['treatment'] == 'exclude_suspect_benign']
display(suspect[['dataset', 'family', 'raw_recall_reference', 'raw_recall_treated', 'mean_diff', 'ci_low', 'ci_high']].round(3))
display(main[main['treatment'].isin(['reference', 'exclude_suspect_benign'])
             & (main['dataset'] == 'cicids2017')].pivot_table(
    index='family', columns='treatment', values=['raw_fpr', 'raw_benign_rows']).round(4))

print('8. Secondary: raw versus deduplicated recall, reference treatment')
display(reference.groupby(['dataset', 'family'])[
    ['raw_recall', 'recall', 'raw_recall_at_oracle_fpr', 'raw_fpr', 'raw_conflicting_share', 'raw_attack_rows']
].mean().round(3))
print('Recall by whether a test flow matches a training attack apart from timing, reference treatment')
display(reference.groupby(['dataset', 'family'])[
    ['signature_matched_rows', 'n_attack', 'recall_signature_matched', 'recall_signature_unmatched']].mean().round(3))

print('Shortcut-dependent results: mean paired raw-recall drop of at least 0.10, with the 95% interval above zero')
display(effects[effects['shortcut_dependent']][EFFECT_COLUMNS].round(3))
print(f"Comparisons under the rule: {rule['n_comparisons']} {rule['n_comparisons_by_dataset']}, with no "
      f"multiple-comparison correction. {rule['n_shortcut_dependent']} flagged.")
print(f"{rule['n_not_testable']} comparisons cannot trigger the rule, because reference recall is already under 0.10:")
display(effects[(effects['role'] == 'rule') & ~effects['rule_testable']][['dataset', 'family', 'treatment', 'raw_recall_reference']].round(3))
print('Flagged on deduplicated rows instead (secondary):')
display(effects[effects['dedup_shortcut_dependent']][
    ['dataset', 'family', 'treatment', 'recall_reference', 'recall_treated', 'dedup_mean_diff', 'dedup_ci_low', 'dedup_ci_high']].round(3))
print('With five seeds the paired test cannot reach p < 0.05. Read the intervals and effect sizes.')


## 8. Package the results

In [ ]:
import zipfile

archive = f'{OUTPUT_DIR}/aegis_artifact_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ('manifest.json', 'results.csv', 'treatment_effects.csv', 'shortcut_rule.json'):
        bundle.write(f'{OUTPUT_DIR}/{name}', name)
print('Return this file with the executed notebook:', archive)
